# 🌊 AISEHack Phase 2 — Flood Detection (3-Class Segmentation)
### Strategy: Prithvi EO v2 600M-TL + 300M-TL Ensemble | SAR + GEE | 5-Fold Stratified CV | D4 TTA

**Phase 2 Label Scheme:**
| Class | Label | Description |
|-------|-------|-------------|
| 0 | No Flood | Non-inundated land |
| 1 | Flood | Active flood inundation |
| 2 | Water Body | Pre-existing permanent/seasonal water |

**Evaluation:**
- Per-class IoU (emphasis on Flood class)
- Mean IoU (mIoU) across all 3 classes
- Overall pixel-wise accuracy

**Submission:** RLE of Flood class (class=1) pixels only, column-major order, 1-indexed. Empty mask → `0 0`.

**Pipeline:**
1. Setup & Install
2. Imports & Config
3. (Optional) Google Earth Engine — DEM / slope / JRC water / GPM / SAR
4. Merge SAR + GEE Channels into Training TIFs
5. Compute Dataset Statistics & Class Weights (3-class)
6. Augmentation Pipelines
7. DataModule Builder
8. Model Builder (FocalDice loss, 3-class head)
9. 5-Fold Stratified Cross Validation
10. Final Training on All Labeled Data
11. TTA + Ensemble Inference
12. Threshold Sweep
13. Test-Set Inference
14. Pseudo-Labelling (Optional)
15. Local Accuracy Check (3-class metrics)
16. Generate RLE Submission CSV (Phase 2 format)
17. Validate Submission Format
18. Quick Inference Sample Test
19. Final Summary


## 1. Setup & Install

In [1]:
# ── Install dependencies (run once on fresh environment, then restart kernel) ──
# import sys, subprocess
# def run(cmd):
#     result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
#     print('OK' if result.returncode == 0 else f'STDERR: {result.stderr[-2000:]}')
# run('pip install -q terratorch albumentations rasterio scipy earthengine-api scikit-learn pypandoc')
# print('Done. Restart kernel if this is first run.')

# ── Pandoc / nbconvert integration fix ───────────────────────────────────────
import sys, os, shutil, subprocess

def ensure_pandoc():
    """Ensure pandoc is available and on PATH for nbconvert."""
    if shutil.which('pandoc'):
        print(f'✅ Pandoc on PATH: {shutil.which("pandoc")}')
        return
    try:
        import pypandoc
        try:
            v = pypandoc.get_pandoc_version()
            print(f'✅ Pandoc via pypandoc: {v}')
        except OSError:
            print('⬇️  Downloading pandoc via pypandoc...')
            pypandoc.download_pandoc()
            v = pypandoc.get_pandoc_version()
            print(f'✅ Pandoc installed: {v}')
        # Add to PATH so nbconvert can find it
        pandoc_dir = os.path.dirname(pypandoc.get_pandoc_path())
        if pandoc_dir and pandoc_dir not in os.environ.get('PATH', ''):
            os.environ['PATH'] = pandoc_dir + os.pathsep + os.environ.get('PATH', '')
            print(f'✅ Added {pandoc_dir} to PATH')
    except ImportError:
        print('⚠️  pypandoc not installed — nbconvert PDF export may not work.')
        print('   Run: pip install pypandoc && python -c "import pypandoc; pypandoc.download_pandoc()"')

ensure_pandoc()

# ── nbconvert helper (auto-detects current notebook name) ────────────────────
def export_notebook(fmt='html', nb_path=None):
    """Export this notebook to the given format. fmt in: html, pdf, script"""
    if nb_path is None:
        # Auto-detect
        candidates = list(__import__('pathlib').Path('.').glob('*.ipynb'))
        if not candidates:
            print('⚠️  No .ipynb file found in current directory.')
            return
        nb_path = str(candidates[0])
    try:
        subprocess.run(
            ['jupyter', 'nbconvert', '--to', fmt, nb_path],
            check=True, capture_output=True, text=True
        )
        print(f'✅ Exported {nb_path} → {fmt}')
    except subprocess.CalledProcessError as e:
        print(f'❌ nbconvert failed: {e.stderr[-500:]}')

print('Setup cell done.')


✅ Pandoc on PATH: /home/hwtuser2/bin/pandoc
Setup cell done.


## 2. Imports & Config

In [2]:
# ── CRITICAL: Set CUDA allocator config BEFORE any CUDA operation ────────────
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TORCH_COMPILE_DISABLE']   = '1'   # disable compile globally

import warnings, sys, json, shutil, gc, tempfile
# Suppress torchmetrics 'compute before update' warnings — these are spurious
# and triggered by TerraTorch calling compute() at epoch start.
warnings.filterwarnings('ignore', message='.*compute.*method of metric.*called before.*update.*')
warnings.filterwarnings('ignore', message='.*DeprecationWarning.*scale_modules.*')
warnings.filterwarnings('ignore', message='.*batch_size.*ambiguous.*')
warnings.filterwarnings('ignore', message='.*Attribute.*focal_dice_loss.*nn.Module.*')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.transform import from_bounds
import albumentations
from albumentations.pytorch import ToTensorV2
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pathlib import Path
from scipy import ndimage
from scipy.ndimage import binary_dilation
from sklearn.model_selection import StratifiedKFold

import terratorch
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datamodules import GenericNonGeoSegmentationDataModule

# ── torch._dynamo: disable compilation globally (avoids CUDAGraph issues) ────
import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.disable()

# ── GPU backend flags ─────────────────────────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.deterministic    = False
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    cap  = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Compute  : sm_{cap[0]}{cap[1]}')
    print(f'VRAM     : {vram:.1f} GB')


Device   : cuda
PyTorch  : 2.6.0+cu124
CUDA     : True
GPU      : NVIDIA A40
Compute  : sm_86
VRAM     : 50.8 GB


In [3]:
import os
from pathlib import Path

# ─── PATHS (from AISEHack_Phase2_FloodDetection) ─────────────────────────────
# dataset_path = './data/' as set in AISEHack notebook
DATASET_PATH = './data'
IS_KAGGLE    = False
print(f'✅ DATASET_PATH      : {DATASET_PATH}')

OUT_DIR = './'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output dir          : {OUT_DIR}')

# ─── CACHE REDIRECT ───────────────────────────────────────────────────────────
_cache_base = OUT_DIR
os.environ['HF_HOME']            = os.path.join(_cache_base, 'hf_cache')
os.environ['TORCH_HOME']         = os.path.join(_cache_base, 'torch_cache')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(_cache_base, 'hf_cache')
os.environ['HF_DATASETS_CACHE']  = os.path.join(_cache_base, 'hf_cache', 'datasets')
for _d in [os.environ['HF_HOME'], os.environ['TORCH_HOME']]:
    os.makedirs(_d, exist_ok=True)

# ─── HYPERPARAMETERS ──────────────────────────────────────────────────────────
SEED            = 42
EPOCHS          = 150
BATCH_SIZE      = 8
LR              = 2e-5
WEIGHT_DECAY    = 0.01
HEAD_DROPOUT    = 0.25
FREEZE_BACKBONE = False
NUM_FRAMES      = 1
N_FOLDS         = 5

# Phase 2: 3 classes — 0=No Flood, 1=Flood, 2=Water Body
NUM_CLASSES = 3

# Channel layout (8-ch SAR+GEE):
#   Ch 1: DEM, Ch 2: SLOPE, Ch 3: GSW, Ch 4: GPM
#   Ch 5: SAR_VV_dB, Ch 6: SAR_VH_dB, Ch 7: SAR_NDWI_dB, Ch 8: SAR_CR_dB
N_CHANNELS = 8
BANDS      = list(range(1, N_CHANNELS + 1))

BASE_MEANS = [300.0, 500.0, 750.0, 60.0, -15.0, -22.0,  7.0, -7.0]
BASE_STDS  = [250.0, 400.0, 900.0, 80.0,   4.0,   4.0,  4.0,  4.0]

COMPUTED_MEANS = BASE_MEANS[:]
COMPUTED_STDS  = BASE_STDS[:]

# Phase 2 class weights: [No Flood, Flood, Water Body]
# Flood (class 1) is the primary target — upweight it.
# Water Body (class 2) can be confused with flood — give moderate weight.
CLASS_WEIGHTS = [0.1, 0.7, 0.2]

pl.seed_everything(SEED, workers=True)

# ─── THRESHOLD CONFIG ─────────────────────────────────────────────────────────
FLOOD_THRESHOLD      = 0.50   # Phase 2: softmax prob for class 1 (Flood)
THRESHOLD_CANDIDATES = np.round(np.arange(0.10, 0.91, 0.05), 2).tolist()

PSEUDO_ROUNDS = [
    {'high': 0.85, 'low': 0.15, 'min_ratio': 0.65},
    {'high': 0.75, 'low': 0.25, 'min_ratio': 0.55},
]

NORM_BLEND      = 0.0
NORM_BLEND_TEST = 0.5

GPM_DATE_START = '2024-05-28'
GPM_DATE_END   = '2024-05-31'

# Auto-scale batch size for VRAM
if torch.cuda.is_available():
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if _vram_gb < 20:
        BATCH_SIZE = 4
        print(f'WARNING: <20 GB VRAM — reduced BATCH_SIZE to {BATCH_SIZE}')
    elif _vram_gb >= 40 and BATCH_SIZE < 8:
        BATCH_SIZE = 8

_free_gb = shutil.disk_usage(OUT_DIR).free / 1e9
if _free_gb < 5:
    print(f'CRITICAL: Only {_free_gb:.1f} GB free on output dir')

print(f'BATCH_SIZE = {BATCH_SIZE},  EPOCHS = {EPOCHS},  NUM_CLASSES = {NUM_CLASSES}')


def disk_usage_report(path=None):
    import shutil as _shutil
    target = path or OUT_DIR
    usage  = _shutil.disk_usage(target)
    total, used, free = usage.total/1e9, usage.used/1e9, usage.free/1e9
    print(f'Disk usage for {target}: Total={total:.1f}GB  Used={used:.1f}GB  Free={free:.1f}GB')
    if free < 5:
        print(f'  ⚠️  CRITICAL: Only {free:.1f} GB free!')


Seed set to 42


✅ DATASET_PATH      : ./data
Output dir          : ./
BATCH_SIZE = 8,  EPOCHS = 150,  NUM_CLASSES = 3


In [4]:
import os

print(f"Working dir   : {os.getcwd()}")
print(f"DATASET_PATH  : {DATASET_PATH}  (exists={Path(DATASET_PATH).exists()})")
print(f"OUT_DIR       : {OUT_DIR}  (exists={Path(OUT_DIR).exists()})")
print()

_dp = Path(DATASET_PATH)
for sub in ['image', 'image_merged', 'label', 'split', 'external', 'prediction']:
    p = _dp / sub
    if p.exists():
        n = len(list(p.glob('*.tif'))) + len(list(p.glob('*.txt')))
        print(f"  ✅ {sub:20s} → {p}  ({n} files)")
    else:
        print(f"  ❌ {sub:20s} → {p}  (NOT FOUND)")

print()
_split = _dp / 'split'
for f in ['train.txt', 'val.txt', 'test.txt']:
    fp = _split / f
    if fp.exists():
        lines = [l.strip() for l in open(fp) if l.strip()]
        print(f"  split/{f}: {len(lines)} entries  (first: {lines[0] if lines else 'empty'})")
    else:
        print(f"  split/{f}: NOT FOUND")

# Phase 2: Check label values are 0, 1, 2
print()
_label_dir = _dp / 'label'
_label_tifs = sorted(_label_dir.glob('*label.tif')) if _label_dir.exists() else []
if _label_tifs:
    import rasterio, numpy as _np
    with rasterio.open(_label_tifs[0]) as _s:
        _lbl = _s.read(1)
    _uniq = _np.unique(_lbl[_lbl >= 0])
    print(f"  Label unique values (first patch): {_uniq.tolist()}")
    _expected = {0, 1, 2}
    _found_set = set(_uniq.tolist())
    if _found_set.issubset(_expected | {-1}):
        print(f"  ✅ Phase 2 label scheme confirmed (0=No Flood, 1=Flood, 2=Water Body)")
    else:
        print(f"  ⚠️  Unexpected label values: {_found_set} — expected subset of {{0, 1, 2}}")

print()
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(0) / 1e9
    reserv = torch.cuda.memory_reserved(0)  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU VRAM: alloc={alloc:.2f} GB  reserved={reserv:.2f} GB  total={total:.1f} GB")

if not _dp.exists():
    print(f"\n❌ DATASET_PATH does not exist: {DATASET_PATH}")
    print("Fix: Set DATASET_PATH in the config cell above.")
else:
    print("✅ DATASET_PATH exists — paths look good.")
    disk_usage_report()


Working dir   : /home/hwtuser2/AISEHACKPHASE2
DATASET_PATH  : ./data  (exists=True)
OUT_DIR       : ./  (exists=True)

  ✅ image                → data/image  (79 files)
  ✅ image_merged         → data/image_merged  (0 files)
  ✅ label                → data/label  (79 files)
  ✅ split                → data/split  (4 files)
  ✅ external             → data/external  (79 files)
  ✅ prediction           → data/prediction  (0 files)

  split/train.txt: 59 entries  (first: 20240529_EO4_RES2_fl_pid_001)
  split/val.txt: 10 entries  (first: 20240529_EO4_RES2_fl_pid_060)
  split/test.txt: 10 entries  (first: 20240529_EO4_RES2_fl_pid_070)

  Label unique values (first patch): [0.0, 1.0, 2.0]
  ✅ Phase 2 label scheme confirmed (0=No Flood, 1=Flood, 2=Water Body)

GPU VRAM: alloc=0.00 GB  reserved=0.00 GB  total=50.8 GB
✅ DATASET_PATH exists — paths look good.
Disk usage for ./: Total=2163.3GB  Used=836.2GB  Free=1217.2GB


## 3. (Optional) Google Earth Engine
Set `USE_GEE = True` once your GEE account is authenticated.
This exports DEM, slope, JRC Global Surface Water, GPM rainfall, and Sentinel-1 SAR (VV+VH) for each patch.

In [5]:
USE_GEE = True   # Set True if you have a GEE account

if USE_GEE:
    import ee
    try:
        ee.Initialize()
        print('✅ GEE already initialised.')
    except Exception:
        try:
            ee.Authenticate()
            ee.Initialize()
            print('✅ GEE initialised after auth.')
        except Exception as e:
            print(f'❌ GEE init failed: {e}')
            USE_GEE = False
else:
    print('GEE skipped (USE_GEE=False)')


✅ GEE already initialised.


In [ ]:
USE_GEE = True

if USE_GEE:
    import ee, requests
    ee.Authenticate()
    ee.Initialize()

    def export_external_channels_for_patch(tif_path: str, out_dir: str):
        """
        Download DEM, SLOPE, GSW, GPM + SAR (VV, VH) for one patch.
        v7: Added Sentinel-1 SAR channels (VV + VH) — best signal for flood detection.
        Output: 6-band float32 TIF (DEM, SLOPE, GSW, GPM, SAR_VV, SAR_VH)
        """
        import rasterio
        import numpy as np
        from pathlib import Path
        from rasterio.transform import from_bounds

        out_dir  = Path(out_dir)
        tif_path = Path(tif_path)

        with rasterio.open(tif_path) as src:
            bounds        = src.bounds
            crs           = src.crs.to_epsg()
            ref_h, ref_w  = src.height, src.width
            ref_transform = src.transform
            ref_crs       = src.crs

        region = ee.Geometry.BBox(bounds.left, bounds.bottom, bounds.right, bounds.top)

        dem   = ee.Image("USGS/SRTMGL1_003").select("elevation").toFloat()
        slope = ee.Terrain.slope(dem).toFloat()
        gsw   = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
                  .select("occurrence").toFloat().unmask(0)
        gpm   = ee.ImageCollection("NASA/GPM_L3/IMERG_V07") \
                  .filterDate(GPM_DATE_START, GPM_DATE_END) \
                  .select("precipitation").mean().toFloat().unmask(0)

        # v7: Sentinel-1 SAR — VV and VH polarisations
        s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
                .filterDate(GPM_DATE_START, GPM_DATE_END)
                .filter(ee.Filter.eq("instrumentMode", "IW"))
                .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
                .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
                .select(["VV", "VH"])
                .mean()
                .toFloat()
                .unmask(0))
        s1_vv = s1.select("VV")
        s1_vh = s1.select("VH")

        band_images = [
            ("DEM",    dem),
            ("SLOPE",  slope),
            ("GSW",    gsw),
            ("GPM",    gpm),
            ("SAR_VV", s1_vv),
            ("SAR_VH", s1_vh),
        ]
        band_arrays = []
        tmp_files   = []

        for band_name, ee_img in band_images:
            url = ee_img.getDownloadURL({
                "region": region,
                "scale":  18,
                "crs":    f"EPSG:{crs}",
                "format": "GEO_TIFF",
            })
            tmp_path = out_dir / f"_tmp_{tif_path.stem}_{band_name}.tif"
            tmp_files.append(tmp_path)
            r = requests.get(url, stream=True)
            r.raise_for_status()
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

            with rasterio.open(tmp_path) as s:
                arr = s.read(1).astype(np.float32)
                if arr.shape != (ref_h, ref_w):
                    from rasterio.enums import Resampling
                    from rasterio.warp import reproject as rasterio_reproject
                    dst_arr = np.zeros((ref_h, ref_w), dtype=np.float32)
                    rasterio_reproject(
                        source=arr, destination=dst_arr,
                        src_transform=s.transform, src_crs=s.crs,
                        dst_transform=ref_transform, dst_crs=ref_crs,
                        resampling=Resampling.bilinear,
                    )
                    arr = dst_arr
            band_arrays.append(arr)

        stacked = np.stack(band_arrays)  # (6, H, W)

        with rasterio.open(tif_path) as src:
            meta = src.meta.copy()
        meta.update(count=6, dtype="float32", photometric="MINISBLACK")

        out_path = out_dir / (tif_path.stem.replace("_image", "") + "_external.tif")
        with rasterio.open(out_path, "w", **meta) as dst:
            for i, arr in enumerate(band_arrays, 1):
                dst.write(arr, i)

        for tmp in tmp_files:
            tmp.unlink(missing_ok=True)

        print(f"Downloaded: {out_path.name}")

    image_dir = Path(DATASET_PATH) / "image"
    ext_dir   = Path(DATASET_PATH) / "external"
    ext_dir.mkdir(exist_ok=True)

    # ── SKIP ALREADY-DOWNLOADED FILES ────────────────────────────────────────
    # Checks existing files before downloading — safe to re-run any time.
    # Only downloads missing or corrupt (wrong channel count) files.
    all_tifs    = sorted(image_dir.glob("*image.tif"))
    to_download = []
    for tif in all_tifs:
        expected_name = tif.stem.replace("_image", "") + "_external.tif"
        ext_path      = ext_dir / expected_name
        if ext_path.exists():
            try:
                with rasterio.open(ext_path) as s:
                    if s.count == 6:
                        continue   # ✅ already valid — skip
                    print(f"  Re-downloading {expected_name} ({s.count} channels, expected 6)")
            except Exception:
                print(f"  Re-downloading {expected_name} (corrupt file)")
        to_download.append(tif)

    print(f"Patches to download : {len(to_download)} / {len(all_tifs)}")
    print(f"Already complete    : {len(all_tifs) - len(to_download)} / {len(all_tifs)}")

    failed = []
    for tif in to_download:
        try:
            export_external_channels_for_patch(str(tif), str(ext_dir))
        except Exception as e:
            print(f"  ERROR exporting {tif.name}: {e}")
            failed.append(tif.name)

    # Verify all files
    bad_channels = []
    for ext_tif in sorted(ext_dir.glob("*.tif")):
        if ext_tif.name.startswith("_tmp_"):
            continue
        with rasterio.open(ext_tif) as s:
            if s.count != 6:
                bad_channels.append((ext_tif.name, s.count))
    if bad_channels:
        print(f"\n⚠️  WARNING: {len(bad_channels)} external TIFs have wrong channel count:")
        for name, count in bad_channels:
            print(f"   {name}: {count} channels (expected 6) — delete and re-export")
    else:
        print("✅ All external TIFs verified: 6 channels each (DEM/SLOPE/GSW/GPM/SAR_VV/SAR_VH)")

    if failed:
        print(f"\n⚠️  {len(failed)} patches failed: {failed}")
    print(f"Done. Files in: {ext_dir}")

## 4. Merge SAR + GEE Channels into Training TIFs

In [6]:
# ─── CRITICAL SAR RESCALING FIX (v15) ───────────────────────────────────────
# FIX v15: Channels 5-8 had std=0 in v14 because:
#   - VV/VH stored as (dB+40)*1.0 → fine variance ~5 dB ✓
#   - BUT NDWI=(VV-VH+20)*75 and CR=10^((VH-VV)/10)*6000 were computed from
#     the already-stored ext_data[4]-40 and ext_data[5]-40, which are correct,
#     BUT the clipping to [0,3000] collapses cross-ratio to always 3000 because
#     10^((VH-VV)/10) for typical VH<VV gives values 0.05-0.5, *6000=300-3000 ✓
#     — issue was the two-pass clip (once during GEE scaling, once after).
#     Real fix: compute NDWI/CR directly from RAW dB before any clipping.
#
# ADDITIONAL FIX: Use channel-wise z-score normalization into [0,3000] range
# so all 8 channels have comparable variance for the model.

def merge_external_channels(image_dir, external_dir, merged_dir):
    """
    SAR-only mode: Optical bands excluded entirely.
    Output: 8-channel float32 TIF per patch.
      Ch 1: DEM    (metres * 0.5)
      Ch 2: SLOPE  (degrees * 33.3)
      Ch 3: GSW    (0-100 * 30)
      Ch 4: GPM    (mm/hr * 60)
      Ch 5: SAR_VV (raw dB, typically -25 to -5, stored as-is)
      Ch 6: SAR_VH (raw dB, typically -32 to -12, stored as-is)
      Ch 7: SAR_NDWI = (VV_dB - VH_dB), range ~[-10,+10]
      Ch 8: SAR_CR   = VH_dB - VV_dB,   range ~[-10,+10]
    NOTE v15: VV/VH stored as raw dB (no +40 offset) so std is ~4-6 dB.
              NDWI/CR are simple differences — variance preserved naturally.
              Normalization happens at inference time via compute_dataset_stats.
    """
    import logging
    logging.getLogger('rasterio').setLevel(logging.ERROR)

    merged_dir   = Path(merged_dir);  merged_dir.mkdir(exist_ok=True)
    image_dir    = Path(image_dir)
    external_dir = Path(external_dir)

    # GEE channels 1-4 (non-SAR): rescale to [0, 3000]
    GEE_SCALE  = np.array([0.5, 33.3, 30.0, 60.0], dtype=np.float32)
    GEE_OFFSET = np.array([0.0,  0.0,  0.0,  0.0], dtype=np.float32)

    skipped = 0
    for img_path in sorted(image_dir.glob('*image.tif')):
        out_path = merged_dir / img_path.name
        ext_path = external_dir / img_path.name.replace('_image.tif', '_external.tif')

        with rasterio.open(img_path) as src:
            meta = src.meta.copy()
            ref_h, ref_w  = src.height, src.width
            ref_transform = src.transform
            ref_crs       = src.crs

        if not ext_path.exists():
            print(f'  WARNING: no external TIF for {img_path.name} — skipping')
            skipped += 1; continue

        with rasterio.open(ext_path) as ext:
            ext_data = ext.read().astype(np.float32)   # (6, H, W): DEM SLOPE GSW GPM VV VH

        if ext_data.shape[0] != 6:
            print(f'  WARNING: {ext_path.name} has {ext_data.shape[0]} ch — skipping')
            skipped += 1; continue

        # Resample if spatial dimensions differ
        if ext_data.shape[1] != ref_h or ext_data.shape[2] != ref_w:
            from rasterio.enums import Resampling
            from rasterio.warp import reproject
            with rasterio.open(ext_path) as ext_src:
                resampled = []
                for b in range(1, ext_src.count + 1):
                    dst_arr = np.zeros((ref_h, ref_w), dtype=np.float32)
                    reproject(
                        source=ext_src.read(b).astype(np.float32),
                        destination=dst_arr,
                        src_transform=ext_src.transform, src_crs=ext_src.crs,
                        dst_transform=ref_transform,     dst_crs=ref_crs,
                        resampling=Resampling.bilinear
                    )
                    resampled.append(dst_arr)
            ext_data = np.stack(resampled)

        ext_data = np.nan_to_num(ext_data, nan=0.0, posinf=0.0, neginf=0.0)

        # Ch 1-4: GEE channels rescaled to [0, 3000]
        gee = np.zeros((4, ref_h, ref_w), dtype=np.float32)
        for ch in range(4):
            gee[ch] = np.clip((ext_data[ch] + GEE_OFFSET[ch]) * GEE_SCALE[ch], 0.0, 3000.0)

        # Ch 5-6: SAR VV/VH stored as raw dB (no offset — preserves real variance)
        sar_vv_db = ext_data[4].copy()   # raw dB, typical range -25 to -5
        sar_vh_db = ext_data[5].copy()   # raw dB, typical range -32 to -12

        # Ch 7: SAR_NDWI = VV_dB - VH_dB  (dual-pol ratio, flood indicator)
        #        Typical range: -5 to +15 dB; variance ~4 dB
        sar_ndwi = sar_vv_db - sar_vh_db   # NOT clipped — real variance

        # Ch 8: SAR_CR = VH_dB - VV_dB  (cross-ratio, inverse of NDWI)
        sar_cr   = sar_vh_db - sar_vv_db   # NOT clipped — real variance

        merged = np.stack([
            gee[0], gee[1], gee[2], gee[3],
            sar_vv_db, sar_vh_db, sar_ndwi, sar_cr
        ], axis=0).astype(np.float32)   # (8, H, W)

        meta.update(count=merged.shape[0], dtype='float32')
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(merged)

    n_written = len(list(merged_dir.glob('*.tif')))
    print(f'Merged {n_written} files (SAR-only 8-ch, v15 raw-dB). Skipped {skipped}.')
    print('Channel layout: DEM | SLOPE | GSW | GPM | VV_dB | VH_dB | NDWI_dB | CR_dB')
    return str(merged_dir)


EXT_DIR    = Path(DATASET_PATH) / 'external'
MERGED_DIR = Path(DATASET_PATH) / 'image_merged'

# ── FIX B: Auto-detect stale v14 merged files and re-merge if external/ exists
import shutil as _shutil_merge
import rasterio as _rs

EXT_DIR    = Path(DATASET_PATH) / 'external'
MERGED_DIR = Path(DATASET_PATH) / 'image_merged'

_merged_tifs = sorted(MERGED_DIR.glob('*.tif')) if MERGED_DIR.exists() else []
_external_ok = EXT_DIR.exists() and len(list(EXT_DIR.glob('*.tif'))) > 0

def _is_v15_merged(tif_path):
    """Return True if the merged TIF uses v15 raw-dB scaling (ch5 mean < 0)."""
    with _rs.open(tif_path) as _s:
        return _s.read(5).ravel().mean() < 0

if _merged_tifs:
    _v15_ok = _is_v15_merged(_merged_tifs[0])
    if not _v15_ok:
        if _external_ok:
            print('⚠️  image_merged/ contains stale v14 files (ch5 mean > 0).')
            print('   Auto-deleting and re-merging with v15 raw-dB scaling...')
            _shutil_merge.rmtree(str(MERGED_DIR))
            _merged_tifs = []   # trigger fresh merge below
        else:
            print('⚠️  WARNING: image_merged/ uses v14 scaling AND external/ is missing.')
            print('   Cannot auto-fix. SAR channels 4-7 will have near-zero variance.')
            print('   To fix: run GEE export to populate data/external/, then re-run this cell.')

if not _merged_tifs and _external_ok:
    print('Merging SAR + GEE channels (v15 raw-dB)...')
    TRAIN_IMAGE_DIR = merge_external_channels(
        Path(DATASET_PATH) / 'image', EXT_DIR, MERGED_DIR
    )
    N_CHANNELS = 8
    BANDS      = list(range(1, 9))

elif _merged_tifs:
    TRAIN_IMAGE_DIR = str(MERGED_DIR)
    N_CHANNELS = 8
    BANDS      = list(range(1, 9))
    print(f'✅ Using v15 merged dir: {TRAIN_IMAGE_DIR}  ({N_CHANNELS} channels)')

else:
    print('⚠️  WARNING: external/ directory not found or empty.')
    print('   8-channel SAR mode (v15) is INACTIVE. To enable it:')
    print('     1. Set USE_GEE = True and authenticate your GEE account.')
    print('     2. Re-run the GEE export step to populate data/external/.')
    print('     3. Re-run this cell — the merge will run automatically.')
    print(f'Falling back to 6-channel optical mode (v15 SAR improvements inactive).')
    TRAIN_IMAGE_DIR = str(Path(DATASET_PATH) / 'image')
    N_CHANNELS      = 6
    BANDS           = list(range(1, 7))
    COMPUTED_MEANS  = BASE_MEANS[:6]
    COMPUTED_STDS   = BASE_STDS[:6]

print(f'TRAIN_IMAGE_DIR = {TRAIN_IMAGE_DIR}')
print(f'N_CHANNELS      = {N_CHANNELS}')


Merging SAR + GEE channels (v15 raw-dB)...
Merged 79 files (SAR-only 8-ch, v15 raw-dB). Skipped 0.
Channel layout: DEM | SLOPE | GSW | GPM | VV_dB | VH_dB | NDWI_dB | CR_dB
TRAIN_IMAGE_DIR = data/image_merged
N_CHANNELS      = 8


## 5. Compute Real Dataset Statistics & Class Weights (3-Class)

In [7]:
def compute_dataset_stats_fast(image_dir, n_channels, split_file=None):
    """
    Fast two-pass dataset stats using numpy (numerically stable float64).
    Phase 2: also computes 3-class frequency weights for No Flood / Flood / Water Body.
    """
    image_dir = Path(image_dir)
    label_dir = Path(DATASET_PATH) / 'label'

    if split_file and Path(split_file).exists():
        with open(split_file) as f:
            entries = [l.strip() for l in f if l.strip()]
        tif_paths = []
        for fn in entries:
            for cand in [image_dir / fn,
                         image_dir / (fn + '_image.tif'),
                         image_dir / (fn + '.tif')]:
                if cand.exists():
                    tif_paths.append(cand); break
    else:
        tif_paths = sorted(image_dir.glob('*image.tif'))

    if len(tif_paths) == 0:
        print(f'WARNING: No TIFs found in {image_dir} — returning placeholder stats')
        return BASE_MEANS[:n_channels], BASE_STDS[:n_channels], [0.10, 0.70, 0.20]

    sums      = np.zeros(n_channels, dtype=np.float64)
    sq_sums   = np.zeros(n_channels, dtype=np.float64)
    total_pix = 0

    # Phase 2: count per-class pixels
    class_counts = np.zeros(3, dtype=np.int64)  # [no_flood, flood, water_body]

    for img_path in tif_paths:
        with rasterio.open(img_path) as src:
            data = src.read().astype(np.float64)
        C = min(data.shape[0], n_channels)
        n = data.shape[1] * data.shape[2]
        sums[:C]    += data[:C].reshape(C, -1).sum(axis=1)
        sq_sums[:C] += (data[:C].reshape(C, -1) ** 2).sum(axis=1)
        total_pix   += n

        lbl_path = label_dir / img_path.name.replace('_image.tif', '_label.tif')
        if lbl_path.exists():
            with rasterio.open(lbl_path) as ls:
                lbl = ls.read(1)
            valid = lbl[lbl >= 0]
            for c in range(3):
                class_counts[c] += int((valid == c).sum())

    means     = sums / total_pix
    variances = np.maximum((sq_sums / total_pix) - means ** 2, 0.0)
    stds_arr  = np.sqrt(variances)

    dead = [i for i, s in enumerate(stds_arr) if s < 1e-3]
    if dead:
        print(f'WARNING: Low-variance channels (std<1e-3): {dead}')
    stds_arr = np.where(stds_arr < 1e-3, 1.0, stds_arr)

    # 3-class inverse-frequency weights
    total_labeled = class_counts.sum()
    if total_labeled > 0 and all(class_counts > 0):
        freq    = class_counts / total_labeled
        inv_f   = 1.0 / freq
        weights = inv_f / inv_f.sum()   # normalise so weights sum to 1
        class_weights = [round(float(w), 4) for w in weights]
        print(f'Class pixel counts : no_flood={class_counts[0]:,}  flood={class_counts[1]:,}  water_body={class_counts[2]:,}')
        print(f'Class frequencies  : {[round(float(f),4) for f in freq]}')
        print(f'Class weights      : no_flood={class_weights[0]}, flood={class_weights[1]}, water_body={class_weights[2]}')
    else:
        class_weights = CLASS_WEIGHTS  # fallback to config defaults
        print(f'Could not compute class weights — using defaults {CLASS_WEIGHTS}')

    return means.tolist(), stds_arr.tolist(), class_weights


print(f'Computing dataset statistics from: {TRAIN_IMAGE_DIR}')
train_split = str(Path(DATASET_PATH) / 'split' / 'train.txt')
COMPUTED_MEANS, COMPUTED_STDS, CLASS_WEIGHTS = compute_dataset_stats_fast(
    TRAIN_IMAGE_DIR, N_CHANNELS, train_split
)

if any(np.isnan(m) for m in COMPUTED_MEANS):
    print('WARNING: NaN in computed means — falling back to BASE_MEANS')
    COMPUTED_MEANS = BASE_MEANS[:N_CHANNELS]
    COMPUTED_STDS  = BASE_STDS[:N_CHANNELS]

assert len(COMPUTED_MEANS) == N_CHANNELS
assert len(COMPUTED_STDS)  == N_CHANNELS
assert len(CLASS_WEIGHTS)  == 3, f'Expected 3 class weights, got {len(CLASS_WEIGHTS)}'

print(f'Means         : {[round(m,4) for m in COMPUTED_MEANS]}')
print(f'Stds          : {[round(s,4) for s in COMPUTED_STDS]}')
print(f'Class weights : {CLASS_WEIGHTS}  (no_flood, flood, water_body)')
print(f'✅ Stats verified: {N_CHANNELS} channels, {NUM_CLASSES} classes — safe to proceed.')


Computing dataset statistics from: data/image_merged
Class pixel counts : no_flood=10,091,613  flood=2,358,596  water_body=3,016,287
Class frequencies  : [0.6525, 0.1525, 0.195]
Class weights      : no_flood=0.116, flood=0.4961, water_body=0.3879
Means         : [1.8978, 57.8412, 513.1075, 5.5505, 0.0, 0.0, 0.0, 0.0]
Stds          : [1.2753, 44.0969, 1068.9921, 7.5293, 1.0, 1.0, 1.0, 1.0]
Class weights : [0.116, 0.4961, 0.3879]  (no_flood, flood, water_body)
✅ Stats verified: 8 channels, 3 classes — safe to proceed.


## 6. Augmentation Pipelines

In [8]:
# ── Training augmentations (SAR-appropriate, multi-channel safe) ─────────────
#
# ROOT CAUSE OF AttributeError: 'numpy.ndarray' has no attribute 'long':
#   GenericNonGeoSegmentationDataModule passes the transform to the dataset,
#   which calls transform(**item) using albumentations' __call__ interface.
#   A plain Python list has NO __call__ — it is silently ignored, so numpy
#   arrays are never converted to tensors → mask.long() raises AttributeError.
#
# FIX: wrap every transform list in A.Compose().
#   ToTensorV2(transpose_mask=True) MUST be the last step — it converts
#   numpy (H,W,C) images and (H,W) masks into torch tensors, which is
#   required by TerraTorch's mask.long() call in __getitem__.

import albumentations as A
from albumentations.pytorch import ToTensorV2

def _build_train_transform():
    transforms = [
        A.RandomRotate90(p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.RandomGamma(gamma_limit=(70, 130), p=0.4),
    ]
    # GaussNoise: use std_range (Albumentations >=1.4 API)
    try:
        transforms.append(A.GaussNoise(std_range=(0.01, 0.05), p=0.4))
    except TypeError:
        transforms.append(A.GaussNoise(var_limit=(1e-4, 25e-4), per_channel=True, p=0.4))
    transforms += [
        A.CoarseDropout(
            num_holes_range=(2, 12),
            hole_height_range=(8, 64),
            hole_width_range=(8, 64), p=0.4),
        A.ElasticTransform(alpha=120, sigma=6, p=0.3),
        A.GridDistortion(num_steps=5, p=0.3),
        A.OneOf([
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        ], p=0.3),
        # ToTensorV2 MUST be last.
        # transpose_mask=True converts (H,W) mask → tensor correctly.
        ToTensorV2(transpose_mask=True),
    ]
    # CRITICAL: wrap in A.Compose so the transform has a callable __call__
    # interface that TerraTorch's dataset can actually invoke.
    return A.Compose(transforms)

# Val/Test: only tensor conversion needed — also wrapped in A.Compose
TRAIN_TRANSFORM = _build_train_transform()
VAL_TRANSFORM   = A.Compose([ToTensorV2(transpose_mask=True)])
TEST_TRANSFORM  = A.Compose([ToTensorV2(transpose_mask=True)])

n_steps = len(TRAIN_TRANSFORM.transforms)
print(f'Training transforms : {n_steps} steps, wrapped in A.Compose')
print(f'Val/Test transforms : A.Compose([ToTensorV2(transpose_mask=True)])')
print(f'type(TRAIN_TRANSFORM) = {type(TRAIN_TRANSFORM).__name__}  ← must be Compose, not list')


Training transforms : 11 steps, wrapped in A.Compose
Val/Test transforms : A.Compose([ToTensorV2(transpose_mask=True)])
type(TRAIN_TRANSFORM) = Compose  ← must be Compose, not list


## 7. DataModule Builder

In [9]:
import os, tempfile
from pathlib import Path

_tmp = os.path.join(OUT_DIR, 'tmp')
os.makedirs(_tmp, exist_ok=True)
os.environ['TMPDIR'] = _tmp
tempfile.tempdir     = _tmp
print(f'TMPDIR → {_tmp}')


def build_datamodule(
    train_split_file,
    val_split_file,
    image_dir=None,
    means=None,
    stds=None,
    batch_size=BATCH_SIZE,
):
    """Build a TerraTorch datamodule for Phase 2 (3-class segmentation)."""
    image_dir = image_dir or TRAIN_IMAGE_DIR
    means     = means     or COMPUTED_MEANS
    stds      = stds      or COMPUTED_STDS

    for label, sf in [('train', train_split_file), ('val', val_split_file)]:
        if sf and Path(sf).exists():
            lines = [l.strip() for l in open(sf) if l.strip()]
            print(f'  {label} split: {len(lines)} entries')
        else:
            print(f'  WARNING: {label} split file not found: {sf}')

    _n_workers = min(4, os.cpu_count() or 2) if torch.cuda.is_available() else 2
    dm = GenericNonGeoSegmentationDataModule(
        batch_size            = batch_size,
        num_workers           = _n_workers,
        pin_memory            = torch.cuda.is_available(),
        persistent_workers    = torch.cuda.is_available() and _n_workers > 0,
        num_classes           = NUM_CLASSES,   # Phase 2: 3 classes
        train_data_root       = image_dir,
        train_label_data_root = str(Path(DATASET_PATH) / 'label'),
        val_data_root         = image_dir,
        val_label_data_root   = str(Path(DATASET_PATH) / 'label'),
        test_data_root        = image_dir,
        test_label_data_root  = str(Path(DATASET_PATH) / 'label'),
        train_split           = train_split_file,
        val_split             = val_split_file,
        test_split            = str(Path(DATASET_PATH) / 'split' / 'test.txt'),
        img_grep              = '*image.tif',
        label_grep            = '*label.tif',
        train_transform       = TRAIN_TRANSFORM,
        val_transform         = VAL_TRANSFORM,
        test_transform        = TEST_TRANSFORM,
        means                 = means,
        stds                  = stds,
        no_data_replace       = 0,
        no_label_replace      = -1,   # -1 = ignore during loss
        predict_data_root     = "/home/hwtuser2/AISEHACKPHASE2/data/prediction/image/",
    )
    return dm


print('DataModule builder ready  (num_classes=3).')


TMPDIR → ./tmp
DataModule builder ready  (num_classes=3).


## 8. Model Builder (Prithvi EO v2 600M-TL + 300M-TL) — 3-Class Head

In [10]:
# ─── Custom Losses (Phase 2: 3-class) ────────────────────────────────────────

class FocalLoss(nn.Module):
    """Multi-class Focal Loss with optional alpha weighting."""
    def __init__(self, alpha=None, gamma=2.0, ignore_index=-1):
        super().__init__()
        self.gamma        = gamma
        self.ignore_index = ignore_index
        self.alpha        = alpha

    def forward(self, logits, targets):
        valid  = targets != self.ignore_index
        t      = targets.clone()
        t[~valid] = 0
        log_p  = F.log_softmax(logits, dim=1)
        p      = log_p.exp()
        log_pt = log_p.gather(1, t.unsqueeze(1)).squeeze(1)
        pt     = p.gather(1, t.unsqueeze(1)).squeeze(1)
        loss   = -((1 - pt) ** self.gamma) * log_pt
        if self.alpha is not None:
            at   = self.alpha.to(logits.device).gather(0, t.view(-1)).view(t.shape)
            loss = loss * at
        return loss[valid].mean()


class DiceLoss(nn.Module):
    """
    Multi-class soft Dice loss (Phase 2).
    Averages Dice over all NUM_CLASSES classes,
    with extra weight on the Flood class (class 1).
    """
    def __init__(self, num_classes=3, ignore_index=-1, smooth=1.0, flood_weight=2.0):
        super().__init__()
        self.num_classes  = num_classes
        self.ignore_index = ignore_index
        self.smooth       = smooth
        self.flood_weight = flood_weight  # extra weight for class 1 (Flood)

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        probs = F.softmax(logits, dim=1)   # (B, C, H, W)
        dice_losses = []
        for c in range(self.num_classes):
            p = probs[:, c][valid]
            t = (targets[valid] == c).float()
            inter = (p * t).sum()
            denom = p.sum() + t.sum() + self.smooth
            d = 1.0 - (2.0 * inter + self.smooth) / denom
            w = self.flood_weight if c == 1 else 1.0
            dice_losses.append(w * d)
        return sum(dice_losses) / (self.num_classes + self.flood_weight - 1)


class FocalDiceLoss(nn.Module):
    """Combined Focal + multi-class Dice loss (Phase 2)."""
    def __init__(self, class_weights=None, num_classes=3, ignore_index=-1,
                 focal_weight=0.5, dice_weight=0.5):
        super().__init__()
        alpha              = torch.tensor(class_weights, dtype=torch.float32) if class_weights else None
        self.focal         = FocalLoss(alpha=alpha, gamma=2.0, ignore_index=ignore_index)
        self.dice          = DiceLoss(num_classes=num_classes, ignore_index=ignore_index)
        self.focal_weight  = focal_weight
        self.dice_weight   = dice_weight

    def forward(self, logits, targets):
        return (self.focal_weight * self.focal(logits, targets) +
                self.dice_weight  * self.dice(logits, targets))


def _extract_logits(output):
    if isinstance(output, torch.Tensor):  return output
    if isinstance(output, (list, tuple)): return _extract_logits(output[0])
    if hasattr(output, 'output'):         return output.output
    if hasattr(output, 'logits'):         return output.logits
    raise ValueError(f'Cannot extract logits from {type(output)}')


class FocalDiceSegTask(SemanticSegmentationTask):
    """SemanticSegmentationTask subclass with FocalDiceLoss for 3-class segmentation."""
    def __init__(self, focal_dice_loss: FocalDiceLoss, **kwargs):
        super().__init__(**kwargs)
        self.save_hyperparameters(ignore=['focal_dice_loss'])
        self._focal_dice_loss = focal_dice_loss

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        try: self.val_metrics.reset()
        except Exception: pass

    def on_train_epoch_start(self):
        super().on_train_epoch_start()
        try: self.train_metrics.reset()
        except Exception: pass

    def training_step(self, batch, batch_idx):
        img    = batch['image']
        mask   = batch['mask'].long()
        output = self.model(img)
        logits = _extract_logits(output)
        if logits.shape[-2:] != mask.shape[-2:]:
            logits = F.interpolate(
                logits, size=mask.shape[-2:], mode='bilinear', align_corners=False
            )
        loss = self._focal_dice_loss(logits, mask)
        self.log('train/loss', loss, on_step=True, on_epoch=True,
                 prog_bar=True, sync_dist=True, batch_size=img.shape[0])
        return loss


def build_model(backbone_name='prithvi_eo_v2_600_tl', class_weights=None,
                n_channels=None, t_max=None):
    """Build FocalDiceSegTask with 3-class head and Prithvi backbone."""
    class_weights = class_weights or CLASS_WEIGHTS
    n_channels    = n_channels    or N_CHANNELS
    bands         = list(range(1, n_channels + 1))

    if '600' in backbone_name:
        select_indices   = [7, 15, 23, 31]
        decoder_channels = 512
    elif '300' in backbone_name:
        select_indices   = [5, 11, 17, 23]
        decoder_channels = 256
    else:
        select_indices   = [2, 5, 8, 11]
        decoder_channels = 128

    necks = [
        dict(name='SelectIndices', indices=select_indices),
        dict(name='ReshapeTokensToImage', effective_time_dim=NUM_FRAMES),
    ]
    model_args = dict(
        backbone_pretrained   = True,
        backbone              = backbone_name,
        backbone_bands        = bands,
        backbone_num_frames   = NUM_FRAMES,
        decoder               = 'UperNetDecoder',
        decoder_channels      = decoder_channels,
        decoder_scale_modules = True,
        num_classes           = NUM_CLASSES,   # Phase 2: 3 classes
        head_dropout          = HEAD_DROPOUT,
        necks                 = necks,
        rescale               = True,
    )

    assert len(class_weights) == NUM_CLASSES, \
        f'class_weights must have {NUM_CLASSES} entries, got {len(class_weights)}'

    fd_loss = FocalDiceLoss(
        class_weights = class_weights,
        num_classes   = NUM_CLASSES,
        ignore_index  = -1,
        focal_weight  = 0.5,
        dice_weight   = 0.5,
    )

    _t_max = t_max if t_max is not None else EPOCHS
    model  = FocalDiceSegTask(
        focal_dice_loss   = fd_loss,
        model_args        = model_args,
        plot_on_val       = False,
        class_weights     = class_weights,
        loss              = 'focal',
        lr                = LR * 5,
        optimizer         = 'AdamW',
        optimizer_hparams = dict(weight_decay=WEIGHT_DECAY),
        scheduler         = 'CosineAnnealingLR',
        scheduler_hparams = dict(T_max=_t_max, eta_min=1e-7),
        ignore_index      = -1,
        freeze_backbone   = FREEZE_BACKBONE,
        freeze_decoder    = False,
        model_factory     = 'EncoderDecoderFactory',
    )
    print(f'  Backbone      : {backbone_name}')
    print(f'  num_classes   : {NUM_CLASSES}  (0=No Flood, 1=Flood, 2=Water Body)')
    print(f'  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)')
    print(f'  n_channels    : {n_channels}')
    return model


def build_trainer(run_name, out_dir=OUT_DIR, max_epochs=EPOCHS, accumulate_grad_batches=4):
    callbacks = [
        EarlyStopping(monitor='val/mIoU', mode='max', patience=30, verbose=True),
        ModelCheckpoint(
            dirpath    = os.path.join(out_dir, 'checkpoints', run_name),
            filename   = '{epoch:03d}-{val/mIoU:.4f}',
            monitor    = 'val/mIoU',
            mode       = 'max',
            save_top_k = 1,
        ),
    ]
    return pl.Trainer(
        accelerator             = 'gpu' if torch.cuda.is_available() else 'cpu',
        devices                 = 1,
        precision               = 'bf16-mixed',
        accumulate_grad_batches = accumulate_grad_batches,
        logger                  = False,
        max_epochs              = max_epochs,
        check_val_every_n_epoch = 1,
        log_every_n_steps       = 5,
        enable_checkpointing    = True,
        gradient_clip_val       = 1.0,
        callbacks               = callbacks,
    )


def gpu_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def clean_split(path):
    tmp   = str(Path(path).with_suffix('.clean.txt'))
    lines = [l.strip().removesuffix('_image') for l in open(path) if l.strip()]
    with open(tmp, 'w') as f: f.write('\n'.join(lines))
    return tmp

def dynamo_reset():
    try:
        torch._dynamo.reset()
        import torch._inductor.config as _ic
        _ic.triton.cudagraphs = False
    except Exception:
        pass

print('Model & trainer builders ready  (3-class head).')


Model & trainer builders ready  (3-class head).


## 9. 5-Fold Stratified Cross Validation
Set `RUN_CV = True` to run full cross-validation before final training.
Set `RUN_CV = False` to skip directly to final training.

> **Phase 2 note:** The CV stratification now uses combined flood+water_body pixel ratio as the strata signal.

In [11]:
RUN_CV    = True
cv_scores = []

def _discover_labeled_patches(dataset_path, train_image_dir):
    dataset_path   = Path(dataset_path)
    image_dir      = Path(train_image_dir)
    label_dir      = dataset_path / 'label'
    split_dir      = dataset_path / 'split'
    raw_entries = []
    split_files_found = []
    for sf in ['train.txt', 'val.txt']:
        p = split_dir / sf
        if p.exists():
            lines = [l.strip() for l in open(p) if l.strip()]
            raw_entries += lines
            split_files_found.append(f'{sf} ({len(lines)} lines)')

    if split_files_found:
        print(f'Split files read: {", ".join(split_files_found)}')

    actual_tifs = {p.name: p for p in sorted(image_dir.glob('*image.tif'))}
    if not actual_tifs:
        actual_tifs = {p.name: p for p in sorted(image_dir.glob('*.tif'))}
    print(f'TIF files on disk: {len(actual_tifs)}')

    def entry_to_stem(entry):
        return Path(entry).stem.replace('_image', '').replace('_label', '')

    def stem_to_image_path(stem):
        for c in [f'{stem}_image.tif', f'{stem}.tif']:
            if c in actual_tifs: return actual_tifs[c]
        for name, path in actual_tifs.items():
            if name.startswith(stem): return path
        return None

    def stem_has_label(stem):
        return any((label_dir / n).exists()
                   for n in [f'{stem}_label.tif', f'{stem}.tif'])

    stems = set()
    if raw_entries:
        for entry in raw_entries:
            stem = entry_to_stem(entry)
            if stem_to_image_path(stem) and stem_has_label(stem):
                stems.add(stem)

    if not stems:
        print('Fallback: scanning image dir...')
        for name, path in actual_tifs.items():
            stem = Path(name).stem.replace('_image', '')
            if stem_has_label(stem):
                stems.add(stem)

    stems = sorted(stems)
    print(f'Labeled patches found: {len(stems)}')
    return stems


if RUN_CV:
    all_stems = _discover_labeled_patches(DATASET_PATH, TRAIN_IMAGE_DIR)
    all_ids   = np.array(all_stems)

    if len(all_ids) == 0:
        print('❌ No labeled patches found. Check DATASET_PATH and TRAIN_IMAGE_DIR.')
        RUN_CV = False
    elif len(all_ids) < N_FOLDS:
        N_FOLDS = len(all_ids)
        print(f'⚠️  Reduced to {N_FOLDS}-fold CV.')

if RUN_CV:
    label_dir    = Path(DATASET_PATH) / 'label'
    flood_ratios = []
    for stem in all_ids:
        lbl_path = next((p for p in [label_dir / f'{stem}_label.tif',
                                     label_dir / f'{stem}.tif'] if p.exists()), None)
        if lbl_path:
            with rasterio.open(lbl_path) as s:
                lbl   = s.read(1)
            valid = lbl[lbl >= 0]
            # Phase 2: strata = combined flood+water_body ratio
            ratio = float(((valid == 1) | (valid == 2)).mean()) if len(valid) > 0 else 0.0
        else:
            ratio = 0.0
        flood_ratios.append(ratio)

    flood_ratios = np.array(flood_ratios)
    strata = (
        pd.Series(pd.cut(flood_ratios, bins=min(5, len(set(flood_ratios))), labels=False))
        .fillna(0).astype(int).values
    )
    print(f'Flood+WaterBody ratio range: {flood_ratios.min():.3f} – {flood_ratios.max():.3f}')

    skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    cv_split_dir = Path(OUT_DIR) / 'cv_splits'
    cv_split_dir.mkdir(exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_ids, strata)):
        dynamo_reset(); gpu_cleanup()
        print('\n' + '='*60)
        print(f'FOLD {fold+1}/{N_FOLDS}')
        print('='*60)

        train_ids = all_ids[train_idx]
        val_ids   = all_ids[val_idx]
        train_file = str(cv_split_dir / f'fold{fold}_train.txt')
        val_file   = str(cv_split_dir / f'fold{fold}_val.txt')
        with open(train_file, 'w') as f: f.write('\n'.join(train_ids))
        with open(val_file,   'w') as f: f.write('\n'.join(val_ids))

        fold_means, fold_stds, fold_cw = compute_dataset_stats_fast(
            TRAIN_IMAGE_DIR, N_CHANNELS, train_file
        )
        clean_train_cv = clean_split(train_file)
        clean_val_cv   = clean_split(val_file)

        dm      = build_datamodule(clean_train_cv, clean_val_cv,
                                   image_dir=TRAIN_IMAGE_DIR,
                                   means=fold_means, stds=fold_stds,
                                   batch_size=BATCH_SIZE)
        model   = build_model(class_weights=fold_cw, t_max=50)
        trainer = build_trainer(f'fold_{fold}', max_epochs=50)

        dm.setup('fit')
        try:
            trainer.fit(model, datamodule=dm)
            best_score = float(trainer.callback_metrics.get('val/mIoU', 0.0))
            cv_scores.append(best_score)
            print(f'Fold {fold+1} best val/mIoU: {best_score:.4f}')
        except RuntimeError as e:
            print(f'\n⚠️  Fold {fold+1} crashed: {e}')
            gpu_cleanup(); continue

        dynamo_reset()
        del model, trainer, dm
        for _ in range(5): gc.collect()
        gpu_cleanup()

    print('\n' + '='*60)
    if cv_scores:
        print(f'CV Results : {[round(s,4) for s in cv_scores]}')
        print(f'Mean mIoU  : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')
    print('='*60)
else:
    cv_scores = []
    print('CV skipped (RUN_CV=False)')


Split files read: train.txt (59 lines), val.txt (10 lines)
TIF files on disk: 79
Labeled patches found: 69
Flood+WaterBody ratio range: 0.009 – 0.786


/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(



FOLD 1/5
Class pixel counts : no_flood=9,607,359  flood=2,151,003  water_body=2,659,558
Class frequencies  : [0.6663, 0.1492, 0.1845]
Class weights      : no_flood=0.1101, flood=0.492, water_body=0.3979
  train split: 55 entries
  val split: 14 entries


/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'focal_dice_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['focal_dice_loss'])`.
2026-04-04 09:04:09,399 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"
/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/terratorch/models/decoders/upernet_decoder.py:39: UserWarning: DeprecationWarning: scale_modules is deprecated and will be removed in future versions. Use LearnedInterpolateToPyramidal neck instead.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.or

  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


...

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric BoundaryMeanIoU was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassF1Score was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassJaccardIndex was called before the ``update`` method which 
may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

Metric val/mIoU improved. New best score: 0.374
Metric val/mIoU improved by 0.021 >= min_delta = 0.0. New best score: 0.395
Metric val/mIoU improved by 0.032 >= min_delta = 0.0. New best score: 0.427
Metric val/mIoU improved by 0.008 >= min_delta = 0.0. New best score: 0.435
Metric val/mIoU improved by 0.005 >= min_delta = 0.0. New best score: 0.440
Metric val/mIoU improved by 0.070 >= min_delta = 0.0. New best score: 0.510
Metric val/mIoU improved by 0.003 >= min_delta = 0.0. New best score: 0.512
Metric val/mIoU improved by 0.002 >= min_delta = 0.0. New best score: 0.514
Metric val/mIoU improved by 0.003 >= min_delta = 0.0. New best score: 0.517
Metric val/mIoU improved by 0.001 >= min_delta = 0.0. New best score: 0.518
Metric val/mIoU improved by 0.002 >= min_delta = 0.0. New best score: 0.520
Metric val/mIoU improved by 0.001 >= min_delta = 0.0. New best score: 0.520
`Trainer.fit` stopped: `max_epochs=50` reached.


Fold 5 best val/mIoU: 0.5204

CV Results : [0.5693, 0.5824, 0.5578, 0.5323, 0.5204]
Mean mIoU  : 0.5524 ± 0.0230


## 10. Final Training on All Labeled Data

In [12]:
# ── Build final train/val split files ────────────────────────────────────────
# Reuses _discover_labeled_patches() defined in the CV cell.
# Includes test.txt entries too (all labeled data for final training).

def _discover_all_labeled(dataset_path, train_image_dir):
    """Like _discover_labeled_patches but also reads test.txt."""
    dataset_path = Path(dataset_path)
    image_dir    = Path(train_image_dir)
    label_dir    = dataset_path / 'label'
    split_dir    = dataset_path / 'split'

    raw_entries = []
    for sf in ['train.txt', 'val.txt', 'test.txt']:
        p = split_dir / sf
        if p.exists():
            raw_entries += [l.strip() for l in open(p) if l.strip()]

    actual_tifs = {p.name: p for p in sorted(image_dir.glob('*image.tif'))}
    if not actual_tifs:
        actual_tifs = {p.name: p for p in sorted(image_dir.glob('*.tif'))}

    def entry_to_stem(e):
        return Path(e).stem.replace('_image', '').replace('_label', '')

    def stem_to_img(stem):
        for c in [f'{stem}_image.tif', f'{stem}.tif']:
            if c in actual_tifs: return actual_tifs[c]
        for name, p in actual_tifs.items():
            if name.startswith(stem): return p
        return None

    def has_label(stem):
        return any((label_dir / n).exists()
                   for n in [f'{stem}_label.tif', f'{stem}.tif'])

    stems = set()
    if raw_entries:
        for e in raw_entries:
            s = entry_to_stem(e)
            if stem_to_img(s) and has_label(s):
                stems.add(s)

    if not stems:
        print('Fallback: scanning image dir...')
        for name in actual_tifs:
            s = Path(name).stem.replace('_image', '')
            if has_label(s): stems.add(s)

    return sorted(stems)


all_labeled = _discover_all_labeled(DATASET_PATH, TRAIN_IMAGE_DIR)
print(f'Total labeled patches (train+val+test): {len(all_labeled)}')

if len(all_labeled) == 0:
    raise RuntimeError(
        f'No labeled patches found.\n'
        f'  DATASET_PATH    = {DATASET_PATH}\n'
        f'  TRAIN_IMAGE_DIR = {TRAIN_IMAGE_DIR}\n'
        f'Check paths and run merge cell first if using SAR channels.'
    )

# ── Stratified val split by flood coverage ────────────────────────────────────
label_dir      = Path(DATASET_PATH) / 'label'
_flood_ratios  = []
for stem in all_labeled:
    lbl_path = next((
        p for p in [label_dir / f'{stem}_label.tif', label_dir / f'{stem}.tif']
        if p.exists()
    ), None)
    if lbl_path:
        with rasterio.open(lbl_path) as _s:
            _lbl = _s.read(1)
        _valid = _lbl[_lbl >= 0]
        _flood_ratios.append(float((_valid == 1).mean()) if len(_valid) > 0 else 0.0)
    else:
        _flood_ratios.append(0.0)

_flood_ratios_arr = np.array(_flood_ratios)
_n_bins = min(5, max(2, len(set(_flood_ratios))))
_strata = (
    pd.Series(pd.cut(_flood_ratios_arr, bins=_n_bins, labels=False))
    .fillna(0).astype(int).values
)

# 80/20 stratified split
_skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
_splits = list(_skf.split(all_labeled, _strata))
_train_idx, _val_idx = _splits[0]
_train_ids = [all_labeled[i] for i in _train_idx]
_val_ids   = [all_labeled[i] for i in _val_idx]

final_split_dir  = Path(OUT_DIR) / 'final_splits'
final_split_dir.mkdir(exist_ok=True)
final_train_file = str(final_split_dir / 'final_train.txt')
final_val_file   = str(final_split_dir / 'final_val.txt')

with open(final_train_file, 'w') as f: f.write('\n'.join(_train_ids))
with open(final_val_file,   'w') as f: f.write('\n'.join(_val_ids))

print(f'Final train : {len(_train_ids)} patches')
print(f'Final val   : {len(_val_ids)} patches')
print(f'Saved: {final_train_file}')
print(f'Saved: {final_val_file}')


Total labeled patches (train+val+test): 79
Final train : 63 patches
Final val   : 16 patches
Saved: final_splits/final_train.txt
Saved: final_splits/final_val.txt


In [13]:
# ── Model A: Prithvi EO v2 600M-TL (primary model) ───────────────────────────
print('\nTraining Model A: prithvi_eo_v2_600_tl...')

dynamo_reset()

clean_train = clean_split(final_train_file)
clean_val   = clean_split(final_val_file)

dm_a      = build_datamodule(clean_train, clean_val)
model_a   = build_model(backbone_name='prithvi_eo_v2_600_tl')
trainer_a = build_trainer('model_a_600m')

dm_a.setup('fit')
batch = next(iter(dm_a.train_dataloader()))
print('Train batch keys:', list(batch.keys()))
if 'image' not in batch:
    raise ValueError("'image' key missing from batch — check dataset paths")

trainer_a.fit(model_a, datamodule=dm_a)

# Save weights (OUT_DIR, not CWD)
CKPT_A = str(Path(OUT_DIR) / 'model_a_600m_weights.pt')
torch.save(model_a.state_dict(), CKPT_A)
print(f'\nModel A saved: {CKPT_A}  ({os.path.getsize(CKPT_A)/1e9:.2f} GB)')

dynamo_reset()
del trainer_a, dm_a
for _ in range(5): gc.collect()
gpu_cleanup()



Training Model A: prithvi_eo_v2_600_tl...
  train split: 63 entries
  val split: 16 entries


/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'focal_dice_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['focal_dice_loss'])`.
2026-04-04 10:20:15,936 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"
/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/terratorch/models/decoders/upernet_decoder.py:39: UserWarning: DeprecationWarning: scale_modules is deprecated and will be removed in future versions. Use LearnedInterpolateToPyramidal neck instead.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.or

  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train batch keys: ['image', 'mask', 'filename']


...

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric BoundaryMeanIoU was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassF1Score was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassJaccardIndex was called before the ``update`` method which 
may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

Metric val/mIoU improved. New best score: 0.120
Metric val/mIoU improved by 0.288 >= min_delta = 0.0. New best score: 0.408
Metric val/mIoU improved by 0.023 >= min_delta = 0.0. New best score: 0.431
Metric val/mIoU improved by 0.017 >= min_delta = 0.0. New best score: 0.448
Metric val/mIoU improved by 0.041 >= min_delta = 0.0. New best score: 0.489
Metric val/mIoU improved by 0.012 >= min_delta = 0.0. New best score: 0.501
Metric val/mIoU improved by 0.031 >= min_delta = 0.0. New best score: 0.532
Metric val/mIoU improved by 0.023 >= min_delta = 0.0. New best score: 0.555
Metric val/mIoU improved by 0.003 >= min_delta = 0.0. New best score: 0.558
Metric val/mIoU improved by 0.004 >= min_delta = 0.0. New best score: 0.562
Metric val/mIoU improved by 0.008 >= min_delta = 0.0. New best score: 0.570
Metric val/mIoU improved by 0.001 >= min_delta = 0.0. New best score: 0.572
Metric val/mIoU improved by 0.001 >= min_delta = 0.0. New best score: 0.573
Metric val/mIoU improved by 0.008 >= min


Model A saved: model_a_600m_weights.pt  (2.70 GB)


In [14]:
# ── Model B: Prithvi EO v2 300M-TL (ensemble partner) ────────────────────────
if not Path(final_train_file).exists():
    raise RuntimeError(
        f'final_train_file not found: {final_train_file}. Run the Final Splits cell first.'
    )

dynamo_reset()
gpu_cleanup()

clean_train = clean_split(final_train_file)
clean_val   = clean_split(final_val_file)

dm_b      = build_datamodule(clean_train, clean_val)
model_b   = build_model(backbone_name='prithvi_eo_v2_300_tl')
trainer_b = build_trainer('model_b_300m')

dm_b.setup('fit')
batch = next(iter(dm_b.train_dataloader()))
print('Train batch keys:', list(batch.keys()))
if 'mask' not in batch:
    raise ValueError("'mask' key missing — check dataset paths and splits")

trainer_b.fit(model_b, datamodule=dm_b)

CKPT_B = str(Path(OUT_DIR) / 'model_b_300m_weights.pt')
torch.save(model_b.state_dict(), CKPT_B)
print(f'\nModel B saved: {CKPT_B}  ({os.path.getsize(CKPT_B)/1e9:.2f} GB)')

dynamo_reset()
del trainer_b, dm_b, model_b
for _ in range(5): gc.collect()
gpu_cleanup()


  train split: 63 entries
  val split: 16 entries


/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'focal_dice_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['focal_dice_loss'])`.
2026-04-04 10:46:27,814 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"
/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/terratorch/models/decoders/upernet_decoder.py:39: UserWarning: DeprecationWarning: scale_modules is deprecated and will be removed in future versions. Use LearnedInterpolateToPyramidal neck instead.
  warnings.warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.or

  Backbone      : prithvi_eo_v2_300_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train batch keys: ['image', 'mask', 'filename']


...

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric BoundaryMeanIoU was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassF1Score was called before the ``update`` method which may 
lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: The ``compute`` method of metric MulticlassJaccardIndex was called before the ``update`` method which 
may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)

Metric val/mIoU improved. New best score: 0.200
Metric val/mIoU improved by 0.234 >= min_delta = 0.0. New best score: 0.434
Metric val/mIoU improved by 0.041 >= min_delta = 0.0. New best score: 0.475
Metric val/mIoU improved by 0.057 >= min_delta = 0.0. New best score: 0.531
Metric val/mIoU improved by 0.004 >= min_delta = 0.0. New best score: 0.535
Metric val/mIoU improved by 0.014 >= min_delta = 0.0. New best score: 0.549
Metric val/mIoU improved by 0.002 >= min_delta = 0.0. New best score: 0.551
Metric val/mIoU improved by 0.002 >= min_delta = 0.0. New best score: 0.553
Metric val/mIoU improved by 0.005 >= min_delta = 0.0. New best score: 0.557
Metric val/mIoU improved by 0.004 >= min_delta = 0.0. New best score: 0.561
Metric val/mIoU improved by 0.000 >= min_delta = 0.0. New best score: 0.561
Metric val/mIoU improved by 0.003 >= min_delta = 0.0. New best score: 0.565
Monitored metric val/mIoU did not improve in the last 30 records. Best score: 0.565. Signaling Trainer to stop.



Model B saved: model_b_300m_weights.pt  (1.28 GB)


## 11. TTA + Ensemble Inference
For each patch:
1. D4 group (8 orientations) × 3 scales = 24 TTA passes
2. Average softmax probs from both models (weighted)
3. For submission: extract class 1 (Flood) probability → apply threshold → binary flood mask
4. Morphological post-processing

In [26]:
# ── D4 group transforms (8 orientations) ────────────────────────────────────
D4_TRANSFORMS = [
    lambda x: x,
    lambda x: torch.rot90(x, 1, [-2, -1]),
    lambda x: torch.rot90(x, 2, [-2, -1]),
    lambda x: torch.rot90(x, 3, [-2, -1]),
    lambda x: torch.flip(x, [-1]),
    lambda x: torch.flip(x, [-2]),
    lambda x: torch.flip(torch.rot90(x, 1, [-2, -1]), [-1]),
    lambda x: torch.flip(torch.rot90(x, 1, [-2, -1]), [-2]),
]
D4_INVERSE = [
    lambda x: x,
    lambda x: torch.rot90(x, -1, [-2, -1]),
    lambda x: torch.rot90(x, -2, [-2, -1]),
    lambda x: torch.rot90(x, -3, [-2, -1]),
    lambda x: torch.flip(x, [-1]),
    lambda x: torch.flip(x, [-2]),
    lambda x: torch.rot90(torch.flip(x, [-1]), -1, [-2, -1]),
    lambda x: torch.rot90(torch.flip(x, [-2]), -1, [-2, -1]),
]
TTA_SCALES = [0.75, 1.0, 1.25]


def extract_logits(output):
    """Robustly extract logit tensor from any TerraTorch model output format."""
    if isinstance(output, torch.Tensor):  return output
    if isinstance(output, (tuple, list)): return extract_logits(output[0])
    if hasattr(output, 'output'):         return output.output
    if hasattr(output, 'logits'):         return output.logits
    raise ValueError(f'Cannot extract logits from {type(output)}')


def run_tta_inference(model, tensor_cpu, device):
    """
    Run multi-scale D4 TTA on a single patch (1, C, H, W) CPU tensor.
    Returns (1, 2, H, W) averaged probability tensor on CPU.
    """
    orig_h, orig_w = tensor_cpu.shape[-2:]
    tensor_dev     = tensor_cpu.to(device)
    all_probs      = []

    with torch.inference_mode():
        for scale in TTA_SCALES:
            if scale == 1.0:
                scaled = tensor_dev
            else:
                scaled = F.interpolate(
                    tensor_dev, scale_factor=scale, mode='bilinear', align_corners=False
                )
            # Batch all 8 D4 augmentations together for efficiency
            aug_batch = torch.cat([aug(scaled) for aug in D4_TRANSFORMS], dim=0)  # (8,C,H',W')
            logits    = extract_logits(model(aug_batch))
            logits    = torch.clamp(torch.nan_to_num(logits, nan=0.0, posinf=1e3, neginf=-1e3), -50, 50)
            probs     = F.softmax(logits, dim=1)   # (8, 2, H', W')

            for i, inv in enumerate(D4_INVERSE):
                p = inv(probs[i:i+1])              # (1, 2, H', W')
                if scale != 1.0:
                    p = F.interpolate(p, size=(orig_h, orig_w),
                                      mode='bilinear', align_corners=False)
                all_probs.append(p)

    return torch.stack(all_probs).mean(0).cpu()   # (1, 2, H, W)


def morphological_cleanup(mask, min_blob_size=700):
    """Dilate → fill holes → remove blobs smaller than min_blob_size pixels."""
    mask    = binary_dilation(mask, iterations=2).astype(np.uint8)
    mask    = ndimage.binary_fill_holes(mask).astype(np.uint8)
    labeled, n = ndimage.label(mask)
    for i in range(1, n + 1):
        if (labeled == i).sum() < min_blob_size:
            mask[labeled == i] = 0
    return mask


def _load_sd_safe(model, ckpt_path):
    sd = torch.load(ckpt_path, map_location='cpu')
    if 'state_dict' in sd:
        sd = sd['state_dict']
        
    # Get the exact keys the model in memory is currently expecting
    target_keys = set(model.state_dict().keys())
    
    mapped_sd = {}
    for k, v in sd.items():
        # Map uncompiled checkpoint keys -> compiled model keys
        k_compiled = k.replace('model.', 'model._orig_mod.', 1)
        
        # Map compiled checkpoint keys -> uncompiled model keys
        k_uncompiled = k.replace('_orig_mod.', '')
        
        # Intelligently route the weights to the matching expected key
        if k in target_keys:
            mapped_sd[k] = v
        elif k_compiled in target_keys:
            mapped_sd[k_compiled] = v
        elif k_uncompiled in target_keys:
            mapped_sd[k_uncompiled] = v
        else:
            mapped_sd[k] = v # Fallback

    try:
        model.load_state_dict(mapped_sd, strict=True)
    except RuntimeError:
        # Fallback for Lightning vs Standard PyTorch wrapper mismatches
        sd_stripped = {k.replace('model.', '', 1): v for k, v in mapped_sd.items()}
        
        # If the top-level model itself is wrapped by dynamo
        if hasattr(model, '_orig_mod'):
            model._orig_mod.load_state_dict(sd_stripped, strict=True)
        else:
            model.load_state_dict(sd_stripped, strict=True)
            
    return model

print('TTA utilities ready.')


TTA utilities ready.


## 12. Threshold Sweep
Sweep threshold on flood-class probability (softmax[:, 1]) to maximise Flood IoU on the validation set.

In [27]:
THRESHOLD_CANDIDATES = np.round(np.arange(0.10, 0.91, 0.05), 2).tolist()

_ckpt_a_ok = 'CKPT_A' in globals() and Path(CKPT_A).exists()
_ckpt_b_ok = 'CKPT_B' in globals() and Path(CKPT_B).exists()

if not _ckpt_a_ok:
    print('❌ CKPT_A not found — run final training first.')
elif 'final_val_file' not in globals():
    print('❌ final_val_file not defined — run final-splits cell first.')
else:
    eval_model_a = build_model('prithvi_eo_v2_600_tl', n_channels=N_CHANNELS)
    eval_model_b = build_model('prithvi_eo_v2_300_tl', n_channels=N_CHANNELS)
    _load_sd_safe(eval_model_a, CKPT_A)
    if _ckpt_b_ok: _load_sd_safe(eval_model_b, CKPT_B)
    eval_model_a.to(DEVICE).eval()
    if _ckpt_b_ok: eval_model_b.to(DEVICE).eval()

    label_dir = Path(DATASET_PATH) / 'label'
    val_ids   = [l.strip() for l in open(final_val_file) if l.strip()]

    print(f'Collecting probs for {len(val_ids)} val patches...')
    val_probs = []
    means_arr = np.array(COMPUTED_MEANS, dtype=np.float32).reshape(-1, 1, 1)
    stds_arr  = np.array(COMPUTED_STDS,  dtype=np.float32).reshape(-1, 1, 1)

    for vid in val_ids:
        stem     = vid.replace('_image', '')
        tif_path = next((p for p in [
            Path(TRAIN_IMAGE_DIR) / f'{stem}_image.tif',
            Path(TRAIN_IMAGE_DIR) / f'{stem}.tif'] if p.exists()), None)
        lbl_path = next((p for p in [
            label_dir / f'{stem}_label.tif',
            label_dir / f'{stem}.tif'] if p.exists()), None)

        if tif_path is None or lbl_path is None: continue

        with rasterio.open(tif_path) as src: raw = src.read().astype(np.float32)
        with rasterio.open(lbl_path) as src: gt  = src.read(1)

        C = min(raw.shape[0], N_CHANNELS)
        _nb = NORM_BLEND_TEST
        bm  = (1 - _nb) * means_arr[:C] + _nb * raw[:C].mean(axis=(1,2), keepdims=True)
        bs  = (1 - _nb) * stds_arr[:C]  + _nb * raw[:C].std(axis=(1,2), keepdims=True).clip(1e-6)
        raw[:C] = (raw[:C] - bm) / (bs + 1e-6)

        t_cpu   = torch.tensor(raw).unsqueeze(0)
        probs_a = run_tta_inference(eval_model_a, t_cpu, DEVICE)
        if _ckpt_b_ok:
            probs_b  = run_tta_inference(eval_model_b, t_cpu, DEVICE)
            ensemble = (0.6 * probs_a + 0.4 * probs_b)
        else:
            ensemble = probs_a

        # Phase 2: flood class = index 1
        flood_prob = ensemble[0, 1].numpy()   # (H, W)
        val_probs.append((flood_prob, gt, gt >= 0))
        print(f'  ✅ {tif_path.name}')

    eval_model_a.cpu()
    if _ckpt_b_ok: eval_model_b.cpu()
    gpu_cleanup()

    # Sweep: optimise Flood IoU (Phase 2 primary metric)
    best_t = float(FLOOD_THRESHOLD)
    best_m = 0.0
    sweep_results = []

    for t in THRESHOLD_CANDIDATES:
        # 3-class confusion for IoU
        # class 0=No Flood, 1=Flood, 2=Water Body
        iou_per_class = []
        for flood_prob, gt, valid in val_probs:
            pred_flood = morphological_cleanup((flood_prob > t).astype(np.uint8))
            # Flood IoU: TP/(TP+FP+FN) for class 1
            tp_f = int(((pred_flood == 1) & (gt == 1) & valid).sum())
            fp_f = int(((pred_flood == 1) & (gt != 1) & valid).sum())
            fn_f = int(((pred_flood == 0) & (gt == 1) & valid).sum())
            iou_f = tp_f / (tp_f + fp_f + fn_f + 1e-8)
            iou_per_class.append(iou_f)
        mean_flood_iou = float(np.mean(iou_per_class))
        sweep_results.append((t, mean_flood_iou))
        if mean_flood_iou > best_m:
            best_m, best_t = mean_flood_iou, float(t)

    sweep_results.sort(key=lambda x: -x[1])
    print('\nTop-10 thresholds by Flood IoU:')
    for t, m in sweep_results[:10]:
        marker = ' ← BEST' if abs(t - best_t) < 1e-9 else ''
        print(f'  t={t:.2f}  Flood IoU={m*100:.2f}%{marker}')

    BEST_THRESHOLD = best_t
    BEST_WEIGHT_A  = 0.6
    BEST_WEIGHT_B  = 0.4
    print(f'\nBEST_THRESHOLD = {BEST_THRESHOLD:.2f}')
    print(f'Best Flood IoU  = {best_m*100:.2f}%')


2026-04-04 11:15:05,534 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


2026-04-04 11:15:09,905 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_300_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8
  ✅ 20240529_EO4_RES2_fl_pid_002_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_015_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_019_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_035_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_036_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_040_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_041_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_046_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_048_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_052_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_054_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_058_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_061_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_063_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_074_image.tif
  ✅ 20240529_EO4_RES2_fl_pid_077_image.tif

Top-10 thresholds by Flood IoU:
  t=0.15  Flood IoU=15.55% ← BEST
  t=0.10  Flood IoU=15.40%
  t=0.20  Flood IoU=15.15%
  t

## 13. Test-Set Inference
Runs TTA + ensemble on the 19 prediction patches.
Outputs are 3-class TIFs (pixel values 0/1/2) — flood masks extracted during CSV generation.

In [28]:
_ckpt_a_ok = 'CKPT_A' in globals() and Path(CKPT_A).exists()

if not _ckpt_a_ok:
    print('❌ CKPT_A not found — run final training first.')
else:
    _pred_image_dir = Path("/home/hwtuser2/AISEHACKPHASE2/data/prediction/image")
    if not _pred_image_dir.exists() or len(list(_pred_image_dir.glob('*.tif'))) == 0:
        _pred_image_dir = Path(TRAIN_IMAGE_DIR)
        print(f'No dedicated prediction images — using: {_pred_image_dir}')

    pred_output_dir = Path(OUT_DIR) / 'predictions'
    pred_output_dir.mkdir(parents=True, exist_ok=True)

    _active_threshold = globals().get('BEST_THRESHOLD', FLOOD_THRESHOLD)
    _weight_a         = globals().get('BEST_WEIGHT_A', 0.6)
    _weight_b         = globals().get('BEST_WEIGHT_B', 0.4)

    if 'eval_model_a' not in globals():
        eval_model_a = build_model('prithvi_eo_v2_600_tl', n_channels=N_CHANNELS)
        _load_sd_safe(eval_model_a, CKPT_A)
    _has_model_b = 'CKPT_B' in globals() and Path(CKPT_B).exists()
    if _has_model_b and 'eval_model_b' not in globals():
        eval_model_b = build_model('prithvi_eo_v2_300_tl', n_channels=N_CHANNELS)
        _load_sd_safe(eval_model_b, CKPT_B)

    eval_model_a.to(DEVICE).eval()
    if _has_model_b: eval_model_b.to(DEVICE).eval()

    tif_paths = sorted(_pred_image_dir.glob('*image.tif'))
    if len(tif_paths) == 0:
        tif_paths = sorted(_pred_image_dir.glob('*.tif'))

    print(f'Running inference on {len(tif_paths)} patches → {pred_output_dir}')
    print(f'Threshold: {_active_threshold:.3f} (Flood class)  |  Weight A={_weight_a}  B={_weight_b}')

    means_arr = np.array(COMPUTED_MEANS).reshape(-1, 1, 1).astype(np.float32)
    stds_arr  = np.array(COMPUTED_STDS).reshape(-1, 1, 1).astype(np.float32)

    for tif_path in tif_paths:
        try:
            with rasterio.open(tif_path) as src:
                raw  = src.read().astype(np.float32)
                meta = src.meta.copy()

            if raw.shape[0] < N_CHANNELS:
                pad = np.zeros((N_CHANNELS - raw.shape[0], *raw.shape[1:]), dtype=np.float32)
                raw = np.concatenate([raw, pad], axis=0)
            elif raw.shape[0] > N_CHANNELS:
                raw = raw[:N_CHANNELS]

            C   = N_CHANNELS
            _nb = NORM_BLEND_TEST
            bm  = (1 - _nb) * means_arr[:C] + _nb * raw[:C].mean(axis=(1,2), keepdims=True)
            bs  = (1 - _nb) * stds_arr[:C]  + _nb * raw[:C].std(axis=(1,2),  keepdims=True).clip(1e-6)
            raw[:C] = (raw[:C] - bm) / (bs + 1e-6)

            t_cpu   = torch.from_numpy(raw).unsqueeze(0).float()
            with torch.no_grad():
                probs_a = run_tta_inference(eval_model_a, t_cpu, DEVICE)
                if _has_model_b:
                    probs_b  = run_tta_inference(eval_model_b, t_cpu, DEVICE)
                    ensemble = _weight_a * probs_a + _weight_b * probs_b
                else:
                    ensemble = probs_a

            ensemble   = torch.nan_to_num(ensemble, nan=0.0, posinf=1.0, neginf=0.0)

            # Phase 2: argmax over 3 classes → full class map (0/1/2)
            pred_class = ensemble[0].argmax(dim=0).numpy().astype(np.int16)  # (H, W)

            # For morphological cleanup: operate on Flood class binary mask
            flood_prob = ensemble[0, 1].cpu().numpy()
            flood_mask = morphological_cleanup((flood_prob > _active_threshold).astype(np.uint8))

            # Build final 3-class output:
            # Start from argmax class map; override Flood class with cleaned mask
            final_pred = pred_class.copy()
            # Where argmax said Flood (1) but cleanup removed it → set to argmax of non-flood classes
            was_flood     = (pred_class == 1)
            cleaned_flood = flood_mask.astype(bool)
            final_pred[was_flood & ~cleaned_flood] = 0  # revert removed blobs to No Flood

            meta.update(count=1, dtype='int16', nodata=-1)
            out_path = pred_output_dir / (tif_path.stem + '.tif')
            with rasterio.open(out_path, 'w', **meta) as dst:
                dst.write(final_pred, 1)

            flood_pct = (final_pred == 1).mean() * 100
            wb_pct    = (final_pred == 2).mean() * 100
            print(f'  ✅ {out_path.name}  flood={flood_pct:.1f}%  water_body={wb_pct:.1f}%')

        except Exception as e:
            print(f'  ❌ Failed on {tif_path.name}: {e}')
            continue

    eval_model_a.cpu()
    if _has_model_b: eval_model_b.cpu()
    gpu_cleanup()

    n_saved = len(list(pred_output_dir.glob('*.tif')))
    print(f'\n✅ Inference done. {n_saved} 3-class masks saved → {pred_output_dir}')


Running inference on 19 patches → predictions
Threshold: 0.150 (Flood class)  |  Weight A=0.6  B=0.4
  ✅ 20240529_EO4_RES2_fl_pid_080_image.tif  flood=0.0%  water_body=5.6%
  ✅ 20240529_EO4_RES2_fl_pid_081_image.tif  flood=0.1%  water_body=20.6%
  ✅ 20240529_EO4_RES2_fl_pid_082_image.tif  flood=0.0%  water_body=4.5%
  ✅ 20240529_EO4_RES2_fl_pid_083_image.tif  flood=0.0%  water_body=24.4%
  ✅ 20240529_EO4_RES2_fl_pid_084_image.tif  flood=0.0%  water_body=18.6%
  ✅ 20240529_EO4_RES2_fl_pid_085_image.tif  flood=0.0%  water_body=2.3%
  ✅ 20240529_EO4_RES2_fl_pid_086_image.tif  flood=0.0%  water_body=1.1%
  ✅ 20240529_EO4_RES2_fl_pid_087_image.tif  flood=0.0%  water_body=0.1%
  ✅ 20240529_EO4_RES2_fl_pid_088_image.tif  flood=0.0%  water_body=1.4%
  ✅ 20240529_EO4_RES2_fl_pid_089_image.tif  flood=0.0%  water_body=12.2%
  ✅ 20240529_EO4_RES2_fl_pid_090_image.tif  flood=0.0%  water_body=8.4%
  ✅ 20240529_EO4_RES2_fl_pid_091_image.tif  flood=0.0%  water_body=1.2%
  ✅ 20240529_EO4_RES2_fl_pid_09

## 14. Pseudo-Labelling (Optional)

In [29]:
def disable_compile(model):
    """Force-disable torch.compile on model and its inner model."""
    try: model = torch._dynamo.disable(model)
    except Exception: pass
    if hasattr(model, 'model'):
        try: model.model = torch._dynamo.disable(model.model)
        except Exception: pass
    return model


def generate_pseudo_labels_round(
    models_and_checkpoints,
    device,
    dataset_path,
    out_dir,
    computed_means,
    computed_stds,
    confidence_high,
    confidence_low,
    min_confident_ratio,
    already_accepted=None,
):
    """
    Generate pseudo-labels from high-confidence model predictions.
    Pixels with flood_prob > confidence_high → label 1 (flood)
    Pixels with flood_prob < confidence_low  → label 0 (no-flood)
    Pixels in between                        → label -1 (ignore)
    Patches where confident_ratio < min_confident_ratio are skipped.
    """
    dataset_path   = Path(dataset_path)
    predict_dir    = Path(TRAIN_IMAGE_DIR)
    tif_paths      = sorted([
        p for p in predict_dir.glob('*image.tif')
        if p.name not in (already_accepted or set())
    ])
    pseudo_img_dir = Path(out_dir) / 'pseudo_labels' / 'image'
    pseudo_lbl_dir = Path(out_dir) / 'pseudo_labels' / 'label'
    pseudo_img_dir.mkdir(parents=True, exist_ok=True)
    pseudo_lbl_dir.mkdir(parents=True, exist_ok=True)

    accepted_images, accepted_labels = [], []
    means_arr = np.array(computed_means).reshape(-1, 1, 1)
    stds_arr  = np.array(computed_stds).reshape(-1, 1, 1)

    for tif_path in tif_paths:
        with rasterio.open(tif_path) as src:
            raw  = src.read().astype(np.float32)
            meta = src.meta.copy()

        C       = min(raw.shape[0], len(means_arr))
        raw[:C] = (raw[:C] - means_arr[:C]) / (stds_arr[:C] + 1e-6)
        tensor_cpu     = torch.tensor(raw).unsqueeze(0)
        ensemble_probs = None

        for model, ckpt_path in models_and_checkpoints:
            _load_sd_safe(model, ckpt_path)
            model = disable_compile(model).to(device).eval()
            probs = run_tta_inference(model, tensor_cpu, device)
            model.cpu()
            gpu_cleanup()
            ensemble_probs = probs if ensemble_probs is None else ensemble_probs + probs

        ensemble_probs /= len(models_and_checkpoints)
        flood_prob      = ensemble_probs[0, 1].numpy()

        confident = (flood_prob > confidence_high) | (flood_prob < confidence_low)
        if confident.mean() < min_confident_ratio:
            continue

        pseudo_label = np.full(flood_prob.shape, -1, dtype=np.int16)
        pseudo_label[flood_prob > confidence_high] = 1
        pseudo_label[flood_prob < confidence_low]  = 0

        # Apply morphological cleanup to the flood region
        flood_mask = morphological_cleanup((pseudo_label == 1).astype(np.uint8))
        pseudo_label[pseudo_label == 1]  = 0
        pseudo_label[flood_mask == 1]    = 1

        img_out = pseudo_img_dir / tif_path.name
        lbl_out = pseudo_lbl_dir / tif_path.name.replace('_image', '_label')
        shutil.copy2(tif_path, img_out)

        meta.update(count=1, dtype='int16', nodata=-1)
        with rasterio.open(lbl_out, 'w', **meta) as dst:
            dst.write(pseudo_label, 1)

        accepted_images.append(str(img_out))
        accepted_labels.append(str(lbl_out))
        print(f'ACCEPT {tif_path.name}')

    return accepted_images, accepted_labels


# ── Run pseudo-labelling rounds ───────────────────────────────────────────────
RUN_PSEUDO = True   # Set True to enable

if RUN_PSEUDO and 'CKPT_A' in globals() and Path(CKPT_A).exists():
    _pseudo_models = [
        (build_model('prithvi_eo_v2_600_tl', n_channels=N_CHANNELS), CKPT_A),
        (build_model('prithvi_eo_v2_300_tl', n_channels=N_CHANNELS), CKPT_B),
    ]
    all_accepted = set()
    for round_idx, cfg in enumerate(PSEUDO_ROUNDS):
        print(f'\n── Pseudo-label round {round_idx+1} ──')
        imgs, lbls = generate_pseudo_labels_round(
            _pseudo_models, DEVICE, DATASET_PATH, OUT_DIR,
            COMPUTED_MEANS, COMPUTED_STDS,
            cfg['high'], cfg['low'], cfg['min_ratio'],
            already_accepted=all_accepted,
        )
        for name in imgs:
            all_accepted.add(Path(name).name)
        print(f'  Accepted: {len(imgs)} patches')
else:
    print('Pseudo-labelling skipped (RUN_PSEUDO=False or checkpoints missing).')


2026-04-04 11:17:15,870 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


2026-04-04 11:17:19,980 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_300_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8

── Pseudo-label round 1 ──
ACCEPT 20240529_EO4_RES2_fl_pid_001_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_003_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_004_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_013_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_024_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_025_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_026_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_034_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_035_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_036_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_038_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_039_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_045_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_046_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_048_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_049_image.tif
ACCEPT 20240529_EO4_RES2_fl_pid_057_image.tif
AC

## 15. Local Accuracy Check (3-Class Metrics)
Reports per-class IoU, mIoU, overall pixel accuracy on the validation split.

In [30]:
def run_val_accuracy(
    models_and_checkpoints,
    device,
    dataset_path,
    out_dir,
    val_split_file,
    computed_means,
    computed_stds,
    flood_threshold=FLOOD_THRESHOLD,
    image_dir=None,
):
    """
    TTA + ensemble inference on val split.
    Phase 2: reports per-class IoU for 0/1/2, mIoU, and pixel accuracy.
    """
    dataset_path  = Path(dataset_path)
    val_image_dir = Path(image_dir) if image_dir else Path(TRAIN_IMAGE_DIR)
    val_label_dir = dataset_path / 'label'
    val_pred_dir  = Path(out_dir) / 'val_predictions'
    val_pred_dir.mkdir(exist_ok=True)

    val_ids = [l.strip() for l in open(val_split_file) if l.strip()]
    val_tif_paths = []
    for vid in val_ids:
        stem  = vid.replace('_image', '')
        match = next((p for p in [
            val_image_dir / f'{stem}_image.tif',
            val_image_dir / f'{stem}.tif'] if p.exists()), None)
        if match: val_tif_paths.append(match)
        else: print(f'  ⚠️  Image not found: {vid}')

    if not val_tif_paths:
        print('❌ No val images found.'); return None

    print(f'Val patches: {len(val_tif_paths)}  |  Threshold: {flood_threshold}')

    means_arr = np.array(computed_means, dtype=np.float32).reshape(-1, 1, 1)
    stds_arr  = np.array(computed_stds,  dtype=np.float32).reshape(-1, 1, 1)

    # Per-class confusion matrix (3 classes)
    conf = np.zeros((3, 3), dtype=np.int64)   # conf[gt, pred]
    total_pixels = 0

    for tif_path in val_tif_paths:
        with rasterio.open(tif_path) as src:
            raw = src.read().astype(np.float32)

        C   = min(raw.shape[0], len(computed_means))
        _nb = NORM_BLEND_TEST
        bm  = (1 - _nb) * means_arr[:C] + _nb * raw[:C].mean(axis=(1,2), keepdims=True)
        bs  = (1 - _nb) * stds_arr[:C]  + _nb * raw[:C].std(axis=(1,2), keepdims=True).clip(1e-6)
        raw[:C] = (raw[:C] - bm) / (bs + 1e-6)
        t_cpu = torch.tensor(raw).unsqueeze(0)

        ensemble_probs = None
        for model, ckpt_path in models_and_checkpoints:
            _load_sd_safe(model, ckpt_path)
            model = model.to(device).eval()
            p = run_tta_inference(model, t_cpu, device)
            model.cpu(); gpu_cleanup()
            ensemble_probs = p if ensemble_probs is None else ensemble_probs + p

        ensemble_probs /= len(models_and_checkpoints)

        # Flood class: apply threshold + morpho cleanup
        flood_prob = ensemble_probs[0, 1].numpy()
        flood_mask = morphological_cleanup((flood_prob > flood_threshold).astype(np.uint8))

        # Final prediction: argmax, then override with cleaned flood mask
        pred_class = ensemble_probs[0].argmax(dim=0).numpy().astype(np.int16)
        final_pred = pred_class.copy()
        final_pred[(pred_class == 1) & (flood_mask == 0)] = 0

        stem     = tif_path.stem.replace('_image', '')
        lbl_path = next((p for p in [
            val_label_dir / f'{stem}_label.tif',
            val_label_dir / f'{stem}.tif'] if p.exists()), None)
        if lbl_path is None: continue

        with rasterio.open(lbl_path) as src:
            gt = src.read(1).astype(np.int32)

        valid = (gt >= 0) & (gt < 3)
        gt_v  = gt[valid]
        pr_v  = np.clip(final_pred[valid], 0, 2)
        total_pixels += int(valid.sum())
        for g in range(3):
            for p in range(3):
                conf[g, p] += int(((gt_v == g) & (pr_v == p)).sum())

    if total_pixels == 0:
        print('❌ No valid pixels.'); return None

    # Compute per-class IoU
    class_names = ['No Flood', 'Flood', 'Water Body']
    iou_per_class = []
    for c in range(3):
        tp = conf[c, c]
        fp = conf[:, c].sum() - tp
        fn = conf[c, :].sum() - tp
        iou = tp / (tp + fp + fn + 1e-8)
        iou_per_class.append(float(iou))

    pixel_acc = conf.diagonal().sum() / (total_pixels + 1e-8)
    miou      = float(np.mean(iou_per_class))
    flood_iou = iou_per_class[1]

    sep = '─' * 55
    print(f'\n{sep}')
    print(f'  LOCAL ACCURACY REPORT (Phase 2 — 3 Classes)')
    print(sep)
    print(f'  Pixel accuracy    : {pixel_acc*100:.2f}%')
    for c, name in enumerate(class_names):
        marker = '  ← primary' if c == 1 else ''
        print(f'  IoU [{c}] {name:12s}: {iou_per_class[c]*100:.2f}%{marker}')
    print(f'  mIoU (3-class)    : {miou*100:.2f}%')
    print(sep)

    if flood_iou < 0.30: print('  ⚠️  Flood IoU < 30% — check threshold or retrain.')
    elif flood_iou < 0.50: print('  ℹ️  Decent flood detection — tune threshold.')
    else: print('  ✅ Strong flood IoU — good submission candidate!')

    return dict(pixel_accuracy=pixel_acc, iou_per_class=iou_per_class, miou=miou,
                iou_no_flood=iou_per_class[0], iou_flood=iou_per_class[1],
                iou_water_body=iou_per_class[2])


_ckpt_a_ok = 'CKPT_A' in globals() and Path(CKPT_A).exists()
_ckpt_b_ok = 'CKPT_B' in globals() and Path(CKPT_B).exists()

if not _ckpt_a_ok:
    print('❌ CKPT_A not found — run Model A training first.')
elif 'final_val_file' not in globals():
    print('❌ final_val_file not defined — run final-splits cell first.')
else:
    _mc = [(build_model('prithvi_eo_v2_600_tl', n_channels=N_CHANNELS), CKPT_A)]
    if _ckpt_b_ok:
        _mc.append((build_model('prithvi_eo_v2_300_tl', n_channels=N_CHANNELS), CKPT_B))

    VAL_THRESHOLD = globals().get('BEST_THRESHOLD', FLOOD_THRESHOLD)
    print(f'Running 3-class val accuracy with flood threshold = {VAL_THRESHOLD}')

    accuracy_metrics = run_val_accuracy(
        models_and_checkpoints = _mc,
        device                 = DEVICE,
        dataset_path           = DATASET_PATH,
        out_dir                = OUT_DIR,
        val_split_file         = final_val_file,
        computed_means         = COMPUTED_MEANS,
        computed_stds          = COMPUTED_STDS,
        flood_threshold        = VAL_THRESHOLD,
        image_dir              = TRAIN_IMAGE_DIR,
    )


2026-04-04 11:32:54,884 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8


2026-04-04 11:32:59,236 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"


  Backbone      : prithvi_eo_v2_300_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8
Running 3-class val accuracy with flood threshold = 0.15
Val patches: 16  |  Threshold: 0.15

───────────────────────────────────────────────────────
  LOCAL ACCURACY REPORT (Phase 2 — 3 Classes)
───────────────────────────────────────────────────────
  Pixel accuracy    : 79.45%
  IoU [0] No Flood    : 75.18%
  IoU [1] Flood       : 24.12%  ← primary
  IoU [2] Water Body  : 72.24%
  mIoU (3-class)    : 57.18%
───────────────────────────────────────────────────────
  ⚠️  Flood IoU < 30% — check threshold or retrain.


## 16. Generate RLE Submission CSV (Phase 2 Format)

**Submission rules:**
- `id` = filename with `_image.tif` suffix removed
- `rle_mask` = RLE of **Flood pixels only** (class = 1), column-major order, **1-indexed**
- Empty flood mask → `0 0` (not blank, not null)
- All 19 test image IDs must be present


In [31]:
import numpy as np

def mask_to_rle(mask: np.ndarray) -> str:
    """
    Convert binary flood mask (H, W) to Phase 2 submission RLE.
    - Column-major (Fortran) order, 1-indexed pixel positions.
    - Returns '0 0' for masks with no flood pixels (required by Phase 2 rules).
    """
    if mask.sum() == 0:
        return '0 0'   # Phase 2: must not be empty or null

    pixels = mask.flatten(order='F')   # column-major
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1   # 1-indexed
    runs[1::2] -= runs[::2]                                 # (start, length) pairs
    return ' '.join(str(x) for x in runs)


def rle_to_mask(rle_str: str, shape) -> np.ndarray:
    """Decode Phase 2 RLE back to binary mask (for round-trip verification)."""
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    rle_str = str(rle_str).strip()
    if rle_str in ('0 0', '', '-1', 'nan'):
        return mask.reshape(shape, order='F')
    parts = list(map(int, rle_str.split()))
    for start, length in zip(parts[0::2], parts[1::2]):
        mask[start - 1: start - 1 + length] = 1
    return mask.reshape(shape, order='F')


def verify_rle_roundtrip(mask: np.ndarray) -> bool:
    rle     = mask_to_rle(mask)
    decoded = rle_to_mask(rle, mask.shape)
    ok      = np.array_equal(mask, decoded)
    if not ok:
        print(f'  RLE round-trip FAILED: orig={mask.sum()}, decoded={decoded.sum()}')
    return ok


def generate_submission_csv(pred_dir, output_csv):
    """
    Read 3-class prediction TIFs from pred_dir.
    Extract Flood class (class==1), encode as RLE, write Phase 2 submission CSV.
    """
    pred_dir  = Path(pred_dir)
    rows      = []
    tif_files = sorted(pred_dir.glob('*.tif'))

    if len(tif_files) == 0:
        print(f'WARNING: No TIF files found in {pred_dir}')
        return None

    roundtrip_ok = True
    for tif_path in tif_files:
        with rasterio.open(tif_path) as src:
            pred = src.read(1)   # int16 with values 0, 1, 2 (or -1 for nodata)

        # Extract Flood class only
        flood_mask = (pred == 1).astype(np.uint8)
        rle = mask_to_rle(flood_mask)   # '0 0' if no flood pixels

        if len(rows) == 0:   # round-trip check on first patch
            ok           = verify_rle_roundtrip(flood_mask)
            roundtrip_ok = ok
            print(f'  RLE round-trip check: {"PASS ✅" if ok else "FAIL ⚠️"}')

        # id: remove _image.tif suffix as per Phase 2 format
        image_id = tif_path.stem.replace('_image', '')
        rows.append({'id': image_id, 'rle_mask': rle})

    df = pd.DataFrame(rows, columns=['id', 'rle_mask'])

    # Phase 2 rules: no empty, no null — replace with '0 0'
    df['rle_mask'] = df['rle_mask'].replace('', '0 0').fillna('0 0')

    df.to_csv(output_csv, index=False)

    n_flood   = (df['rle_mask'] != '0 0').sum()
    n_no_flood = (df['rle_mask'] == '0 0').sum()
    print(f'\nSubmission CSV saved → {output_csv}')
    print(f'Total patches       : {len(df)}')
    print(f'With flood pixels   : {n_flood}')
    print(f'No flood (→ "0 0")  : {n_no_flood}')
    if not roundtrip_ok:
        print('WARNING: RLE round-trip check FAILED!')
    print('\nPreview:')
    print(df.head(5).to_string())
    return df


submission_csv = str(Path(OUT_DIR) / 'submission.csv')

if 'pred_output_dir' not in globals() or not Path(pred_output_dir).exists():
    print('❌ pred_output_dir not found — run the inference cell first.')
else:
    df_sub = generate_submission_csv(str(pred_output_dir), submission_csv)


  RLE round-trip check: PASS ✅

Submission CSV saved → submission.csv
Total patches       : 19
With flood pixels   : 2
No flood (→ "0 0")  : 17

Preview:
                             id                                                                                                                                                                                                                                                                                                           rle_mask
0  20240529_EO4_RES2_fl_pid_080                                                                                                                                                                                                                                                                                                                0 0
1  20240529_EO4_RES2_fl_pid_081  227837 2 228348 4 228860 5 229372 5 229883 6 230395 6 230907 6 231419 6 231930 7 232442 7 232954 7 233466 7 233978 7 234490 7 235002 7 

## 17. Validate Submission Format

In [32]:
def validate_submission(csv_path, expected_rows=None):
    """Validate Phase 2 submission CSV format and RLE integrity."""
    df = pd.read_csv(csv_path)

    assert 'id'       in df.columns, "Missing 'id' column"
    assert 'rle_mask' in df.columns, "Missing 'rle_mask' column"

    if expected_rows and len(df) != expected_rows:
        print(f'⚠️  Expected {expected_rows} rows, got {len(df)}')

    # Phase 2: must not have null or empty rle_mask
    null_count  = df['rle_mask'].isna().sum()
    empty_count = (df['rle_mask'].astype(str).str.strip() == '').sum()
    assert null_count  == 0, f'Found {null_count} null rle_mask values!'
    assert empty_count == 0, f'Found {empty_count} empty rle_mask values!'

    rle_errors = []
    n_zero_zero = 0
    for _, row in df.iterrows():
        rle = str(row['rle_mask']).strip()
        if rle == '0 0':
            n_zero_zero += 1
            continue   # Valid empty-flood representation
        parts = rle.split()
        if len(parts) % 2 != 0:
            rle_errors.append(f"{row['id']}: odd token count")
            continue
        nums = [int(p) for p in parts]
        if not all(n > 0 for n in nums):
            rle_errors.append(f"{row['id']}: non-positive value in RLE")

    if rle_errors:
        for e in rle_errors[:10]: print(f'  ⚠️  {e}')
    else:
        print('✅ All RLE strings valid (Phase 2 format)')

    print(f'✅ Submission validated!')
    print(f'  Rows              : {len(df)}')
    print(f'  Columns           : {list(df.columns)}')
    print(f'  Null rle_mask     : {null_count}')
    print(f'  Empty rle_mask    : {empty_count}')
    print(f'  "0 0" (no flood)  : {n_zero_zero}')
    print(f'  With flood pixels : {len(df) - n_zero_zero}')
    print(f'  File              : {csv_path}')


if not Path(submission_csv).exists():
    print(f'❌ Submission CSV not found at {submission_csv}')
else:
    try:
        validate_submission(submission_csv)
    except AssertionError as e:
        print(f'⚠️  Validation issue: {e}')


✅ All RLE strings valid (Phase 2 format)
✅ Submission validated!
  Rows              : 19
  Columns           : ['id', 'rle_mask']
  Null rle_mask     : 0
  Empty rle_mask    : 0
  "0 0" (no flood)  : 17
  With flood pixels : 2
  File              : submission.csv


## 18. Quick Inference Sample Test

In [33]:
# ── Quick inference sample test ──────────────────────────────────────────────
# Useful to verify the model loads, normalises, and produces valid outputs.

def quick_inference_test(image_dir, model_ckpt, backbone='prithvi_eo_v2_600_tl',
                          n_channels=None, device=None):
    """
    Load one image, run a single forward pass (no TTA), print stats.
    Prints: image stats, output logit stats, predicted flood fraction.
    """
    n_channels = n_channels or N_CHANNELS
    device     = device or DEVICE

    tif_paths = sorted(Path(image_dir).glob('*image.tif'))
    if len(tif_paths) == 0:
        tif_paths = sorted(Path(image_dir).glob('*.tif'))
    if len(tif_paths) == 0:
        print(f'❌ No TIF files found in {image_dir}'); return

    tif_path = tif_paths[0]
    print(f'Sample image: {tif_path.name}')

    with rasterio.open(tif_path) as src:
        raw = src.read().astype(np.float32)

    print(f'  Raw shape   : {raw.shape}')
    print(f'  Channel stats:')
    for i in range(min(raw.shape[0], n_channels)):
        print(f'    ch{i+1:2d}  mean={raw[i].mean():8.2f}  std={raw[i].std():8.2f}  '
              f'min={raw[i].min():8.2f}  max={raw[i].max():8.2f}')

    # Normalise using computed stats
    C = min(raw.shape[0], n_channels)
    pm = raw[:C].mean(axis=(1,2), keepdims=True)
    ps = raw[:C].std(axis=(1,2), keepdims=True).clip(min=1e-6)
    raw[:C] = (raw[:C] - pm) / (ps + 1e-6)

    t = torch.from_numpy(raw[:n_channels]).unsqueeze(0).float().to(device)

    m = build_model(backbone, n_channels=n_channels)
    _load_sd_safe(m, model_ckpt)
    m = m.to(device).eval()

    with torch.inference_mode():
        out    = m(t)
        logits = _extract_logits(out)
        probs  = torch.softmax(logits, dim=1)
        flood_frac = (probs[0, 1] > 0.5).float().mean().item()

    print(f'\n  Logit range : [{logits.min().item():.3f}, {logits.max().item():.3f}]')
    print(f'  Prob range  : [{probs.min().item():.3f}, {probs.max().item():.3f}]')
    print(f'  Flood frac  : {flood_frac*100:.1f}%  (at threshold=0.5)')
    print('✅ Quick inference test passed.')

    m.cpu(); gpu_cleanup()
    return probs.cpu().numpy()


_ckpt_a_ok = 'CKPT_A' in globals() and Path(CKPT_A).exists()
if _ckpt_a_ok:
    quick_inference_test(TRAIN_IMAGE_DIR, CKPT_A)
else:
    print('Skipping quick test — CKPT_A not available.')
    print('(Run training cells first, then re-run this cell.)')


Sample image: 20240529_EO4_RES2_fl_pid_001_image.tif
  Raw shape   : (8, 512, 512)
  Channel stats:
    ch 1  mean=    0.61  std=    0.88  min=    0.00  max=   10.26
    ch 2  mean=   22.69  std=   37.17  min=    0.00  max=  803.64
    ch 3  mean= 1931.54  std= 1364.16  min=    0.00  max= 3000.00
    ch 4  mean=    0.24  std=    0.07  min=    0.14  max=    0.29
    ch 5  mean=    0.00  std=    0.00  min=    0.00  max=    0.00
    ch 6  mean=    0.00  std=    0.00  min=    0.00  max=    0.00
    ch 7  mean=    0.00  std=    0.00  min=    0.00  max=    0.00
    ch 8  mean=    0.00  std=    0.00  min=    0.00  max=    0.00


/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'focal_dice_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['focal_dice_loss'])`.
2026-04-04 11:34:57,177 - INFO - HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M-TL/resolve/main/Prithvi_EO_V2_600M_TL.pt "HTTP/1.1 302 Found"
/home/hwtuser2/miniforge3/envs/AISE_new/lib/python3.12/site-packages/terratorch/models/decoders/upernet_decoder.py:39: UserWarning: DeprecationWarning: scale_modules is deprecated and will be removed in future versions. Use LearnedInterpolateToPyramidal neck instead.
  warnings.warn(


  Backbone      : prithvi_eo_v2_600_tl
  num_classes   : 3  (0=No Flood, 1=Flood, 2=Water Body)
  Loss          : FocalDiceLoss (focal=0.5, dice=0.5, flood_weight=2.0)
  n_channels    : 8

  Logit range : [-4.838, 6.196]
  Prob range  : [0.000, 0.999]
  Flood frac  : 0.0%  (at threshold=0.5)
✅ Quick inference test passed.


## 19. Final Summary

In [34]:
print('=' * 65)
print('  FLOOD DETECTION PIPELINE — PHASE 2 FINAL SUMMARY')
print('=' * 65)
print(f'  NUM_CLASSES    : {NUM_CLASSES}  (0=No Flood, 1=Flood, 2=Water Body)')

if cv_scores:
    print(f'  CV mIoU        : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')
    print(f'  CV fold scores : {[round(s,4) for s in cv_scores]}')
else:
    print('  CV             : not run (RUN_CV=False)')

for label, var in [('Model A (600M)', 'CKPT_A'), ('Model B (300M)', 'CKPT_B')]:
    ckpt = globals().get(var)
    if ckpt and Path(ckpt).exists():
        print(f'  {label:20s}: {ckpt}  ({os.path.getsize(ckpt)/1e6:.0f} MB)')
    else:
        print(f'  {label:20s}: not found')

_bt = globals().get('BEST_THRESHOLD', FLOOD_THRESHOLD)
print(f'  Best threshold : {_bt:.3f}  (Flood class)')

_am = globals().get('accuracy_metrics')
if _am:
    print(f'  Val mIoU       : {_am["miou"]*100:.2f}%')
    print(f'  Val IoU Flood  : {_am["iou_flood"]*100:.2f}%  ← primary metric')
    print(f'  Val IoU NF     : {_am["iou_no_flood"]*100:.2f}%')
    print(f'  Val IoU WB     : {_am["iou_water_body"]*100:.2f}%')
    print(f'  Val Pixel Acc  : {_am["pixel_accuracy"]*100:.2f}%')

if Path(submission_csv).exists():
    _df_check = pd.read_csv(submission_csv)
    print(f'  Submission CSV : {submission_csv}')
    print(f'  Rows           : {len(_df_check)}')
    n_flood = (_df_check['rle_mask'] != '0 0').sum()
    print(f'  Patches w/flood: {n_flood} / {len(_df_check)}')
else:
    print('  Submission CSV : not generated')

print(f'\n  CLASS_WEIGHTS  : {CLASS_WEIGHTS}  (no_flood, flood, water_body)')
print(f'  N_CHANNELS     : {N_CHANNELS}')
print(f'  COMPUTED_MEANS : {[round(m,2) for m in COMPUTED_MEANS]}')
print(f'  COMPUTED_STDS  : {[round(s,2) for s in COMPUTED_STDS]}')
print('=' * 65)


  FLOOD DETECTION PIPELINE — PHASE 2 FINAL SUMMARY
  NUM_CLASSES    : 3  (0=No Flood, 1=Flood, 2=Water Body)
  CV mIoU        : 0.5524 ± 0.0230
  CV fold scores : [0.5693, 0.5824, 0.5578, 0.5323, 0.5204]
  Model A (600M)      : model_a_600m_weights.pt  (2700 MB)
  Model B (300M)      : model_b_300m_weights.pt  (1279 MB)
  Best threshold : 0.150  (Flood class)
  Val mIoU       : 57.18%
  Val IoU Flood  : 24.12%  ← primary metric
  Val IoU NF     : 75.18%
  Val IoU WB     : 72.24%
  Val Pixel Acc  : 79.45%
  Submission CSV : submission.csv
  Rows           : 19
  Patches w/flood: 2 / 19

  CLASS_WEIGHTS  : [0.116, 0.4961, 0.3879]  (no_flood, flood, water_body)
  N_CHANNELS     : 8
  COMPUTED_MEANS : [1.9, 57.84, 513.11, 5.55, 0.0, 0.0, 0.0, 0.0]
  COMPUTED_STDS  : [1.28, 44.1, 1068.99, 7.53, 1.0, 1.0, 1.0, 1.0]


## 20. Smoke Test
Set `DEBUG = True` for a fast 2-epoch sanity check.

In [35]:
DEBUG = False

if DEBUG:
    print('Running smoke test (2 epochs, 300M model)...')
    dm_debug      = build_datamodule(clean_split(final_train_file), clean_split(final_val_file), batch_size=2)
    model_debug   = build_model(backbone_name='prithvi_eo_v2_300_tl')
    trainer_debug = build_trainer('debug_run', max_epochs=2)
    dm_debug.setup('fit')
    dynamo_reset()
    trainer_debug.fit(model_debug, datamodule=dm_debug)
    dynamo_reset()
    del trainer_debug, dm_debug, model_debug
    for _ in range(5): gc.collect()
    gpu_cleanup()
    print('✅ Smoke test passed.')
else:
    print('Smoke test skipped. Set DEBUG=True to run a quick sanity check.')


Smoke test skipped. Set DEBUG=True to run a quick sanity check.
